# Silent-Failure Detection and Risk Control for Brain-Tumour MRI Classifiers under Domain Shift

This notebook runs the complete experimental pipeline for the revised manuscript. Every number, table and figure is generated from saved result files, so the paper can never disagree with the code.

## What the pipeline does

1. **Discovers datasets automatically** under `/kaggle/input`, by folder structure rather than by hard-coded paths. It supports image-folder datasets (Br35H, Nickparvar BTD) and the original Figshare `.mat` release, which includes patient IDs and tumour masks.
2. **Deduplicates across and within datasets.** It uses perceptual hashing (pHash + dHash on brain-cropped images) with union-find clustering. Cross-domain duplicates are removed (the side is chosen per shift), label-conflicting duplicates are dropped, and every removal is reported.
3. **Makes leakage-free splits.** Splits are stratified and group-aware: near-duplicate clusters, plus patient IDs where available, never straddle two splits. Splits are fixed by `split_seed`; training seeds vary.
4. **Trains each backbone** (CNNs, transformers, and a DINOv2 foundation model) with optional DANN, which uses *unlabelled* target images, and optional MixStyle. Mixed precision, early stopping on source validation only.
5. **Benchmarks 9+ confidence scoring functions on the same classifier:**
   - MSP, entropy, MaxLogit, energy
   - MC-Dropout (MSP and mutual information)
   - deep k-NN feature distance
   - learned failure heads trained post-hoc on frozen features (the original margin/diversity head and a ConfidNet-style TCP head)
   - Deep Ensembles across seeds
6. **Novel contribution: label-free, shift-aware risk control of silent failures.**
   - Conformal risk control of the **silent-failure rate** (SFR = P(wrong and not flagged)) and of the **false-safe rate** (FSR = P(not flagged | wrong)).
   - It is calibrated on labelled source data, then **importance-weighted with a density-ratio model fitted on unlabelled target scans**.
   - It is compared against few-label target calibration (k = 25…200 labelled target scans) and an oracle, so the paper can report whether a stated guarantee *still holds on the new hospital* and what review workload it costs.
7. **Reports with uncertainty.** Seeds give mean ± SD, with bootstrap 95% CIs, Friedman and Wilcoxon (Holm-corrected) tests, CNN-vs-transformer tests, shift-magnitude correlations, an ablation study, per-subtype silent-failure analysis, LaTeX tables and an auto-generated `methods_summary.md`.

## How to run on Kaggle

1. **Add data** (right panel → *Add Input*). Folder names are auto-detected, so dataset slugs don't matter:
   - Br35H: `ahmedhamada0/brain-tumor-detection` (folders `yes/`, `no/`)
   - Nickparvar BTD: `masoudnickparvar/brain-tumor-mri-dataset` (`Training/`, `Testing/`)
   - SARTAJ: `sartajbhuvaji/brain-tumor-classification-mri` (`Training/`, `Testing/` with `glioma_tumor/` …)
   - Figshare (Cheng et al.): any Kaggle upload of the original `.mat` files, or image folders `1/ 2/ 3/` with "figshare" in the path
2. **Settings:** Accelerator **GPU T4 ×1** (or P100), Internet **On** (needed for pretrained weights).
3. Run once with `RUN_MODE = "smoke"` (≈10–20 min) to check that everything works end to end.
4. Switch to `RUN_MODE = "paper"` and use **Save Version → Run All (commit)**. Runs that already finished are skipped automatically.
   - If a session hits the 12-hour limit, add the previous version's output as an input and list its `fdshift` folder in `CFG["resume_dirs"]`. The notebook continues where it stopped.
5. Results are in `/kaggle/working/fdshift/`: `results/tables/*.csv|.tex`, `results/figures/*.png`, `results/methods_summary.md`, and `fdshift_results.zip`.

Everything you might need to change is in the **Configuration** cell. Nothing else should need editing.


## 1 · Environment

In [1]:
# ===== 1. Environment & imports =====
import importlib, subprocess, sys

def _ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg], check=False)

for _p, _pip in [("timm", "timm"), ("h5py", "h5py"), ("sklearn", "scikit-learn"), ("scipy", "scipy")]:
    _ensure(_p, _pip)

import os, re, json, math, time, glob, copy, random, shutil, hashlib, warnings, zipfile, platform
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import scipy.fft
import scipy.stats as st
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image, ImageFile

from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict, StratifiedKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             balanced_accuracy_score, accuracy_score, confusion_matrix)
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

Python 3.13.15 | numpy 2.1.3 | pandas 2.3.3


## 2 · Configuration
Everything adjustable lives here: datasets (auto-discovered), shifts, backbones, seeds, hyper-parameters, risk levels. Start with `RUN_MODE = "smoke"`.

In [2]:
# ===== 2. Configuration — the ONLY cell you should need to edit =====
RUN_MODE = "paper"   # "smoke" = fast end-to-end check (~10-20 min on T4).  "paper" = full experiments.

CFG = {
    # ---------- paths ----------
    "input_root": "/kaggle/input",
    "work_dir": "/kaggle/working/fdshift",
    # Previous outputs to resume from (finished runs are copied in and skipped), e.g.
    # ["/kaggle/input/<your-previous-notebook-output>/fdshift"]
    "resume_dirs": [],
    "max_hours": 11.0,          # stop launching new runs after this wall time (Kaggle limit is 12 h)

    # ---------- datasets: found automatically by folder structure ----------
    # child_dirs : class folders that must exist directly inside the dataset root
    # path_contains : text that must appear in the folder's path below input_root (e.g. Training/ vs Testing/)
    # mat : True for the original Figshare .mat release (patient IDs + masks are read from the files)
    # class_map : optional {folder-or-label -> canonical class}; otherwise "canonical_classes" is used
    # root : set an explicit path to bypass auto-discovery
    "datasets": [
        {"name": "br35h",     "root": None, "find": {"child_dirs": ["yes", "no"]}},
        {"name": "btd_train", "root": None, "find": {"child_dirs": ["glioma", "meningioma", "notumor", "pituitary"],
                                                     "path_contains": "train"}},
        {"name": "btd_test",  "root": None, "find": {"child_dirs": ["glioma", "meningioma", "notumor", "pituitary"],
                                                     "path_contains": "test"}},
        {"name": "sartaj_train", "root": None, "find": {"child_dirs": ["glioma_tumor", "meningioma_tumor", "no_tumor", "pituitary_tumor"],
                                                        "path_contains": "train"}},
        {"name": "sartaj_test",  "root": None, "find": {"child_dirs": ["glioma_tumor", "meningioma_tumor", "no_tumor", "pituitary_tumor"],
                                                        "path_contains": "test"}},
        {"name": "figshare",  "root": None,
         "find": [{"mat": True},                                                   # original .mat release
                  {"child_dirs": ["1", "2", "3"], "path_contains": "figshare"},     # image re-uploads
                  {"child_dirs": ["glioma", "meningioma", "pituitary"], "path_contains": "figshare"}],
         "class_map": {"1": "meningioma", "2": "glioma", "3": "pituitary"}},
    ],
    # normalised folder name (lower-case, letters/digits only) -> canonical class
    "canonical_classes": {
        "no": "notumor", "notumor": "notumor", "notumour": "notumor", "normal": "notumor", "healthy": "notumor",
        "yes": "tumor", "tumor": "tumor", "tumour": "tumor",
        "glioma": "glioma", "gliomatumor": "glioma", "gliomatumour": "glioma",
        "meningioma": "meningioma", "meningiomatumor": "meningioma", "meningiomatumour": "meningioma",
        "pituitary": "pituitary", "pituitarytumor": "pituitary", "pituitarytumour": "pituitary",
    },
    # task -> {canonical class -> label index}; class_names in label order
    "tasks": {
        "binary":  {"labels": {"notumor": 0, "tumor": 1, "glioma": 1, "meningioma": 1, "pituitary": 1},
                    "class_names": ["no_tumour", "tumour"], "positive_class": 1},
        "subtype": {"labels": {"glioma": 0, "meningioma": 1, "pituitary": 2},
                    "class_names": ["glioma", "meningioma", "pituitary"], "positive_class": None},
    },
    # Domain shifts, run in this order (put the most important first).
    # dedup_drop: which side loses images that near-duplicate the other side ("source" keeps the target test intact).
    # A shift is skipped automatically (with a message) if one of its datasets is not attached; "enabled": False skips it.
    # Note: the Nickparvar BTD set is itself a merge of Figshare + SARTAJ + Br35H (its no-tumour class comes from Br35H),
    # so the shifts between independent sources (Br35H, SARTAJ, Figshare) are the scientifically cleanest ones.
    "shifts": [
        {"name": "br35h_to_btd",       "task": "binary",  "source": ["br35h"],    "target": ["btd_test"],                    "dedup_drop": "source"},
        {"name": "br35h_to_sartaj",    "task": "binary",  "source": ["br35h"],    "target": ["sartaj_train", "sartaj_test"], "dedup_drop": "source"},
        {"name": "figshare_to_sartaj", "task": "subtype", "source": ["figshare"], "target": ["sartaj_train", "sartaj_test"], "dedup_drop": "target"},
        {"name": "btd_to_br35h",       "task": "binary",  "source": ["btd_train"], "target": ["br35h"],                     "dedup_drop": "source",
         "enabled": False},
    ],

    # ---------- backbones (timm names); "enabled": False to skip ----------
    "backbones": [
        {"name": "resnet50",                     "family": "CNN",         "lr": 1e-4, "enabled": True},
        {"name": "densenet121",                  "family": "CNN",         "lr": 1e-4, "enabled": True},
        {"name": "efficientnet_b0",              "family": "CNN",         "lr": 1e-4, "enabled": True},
        {"name": "convnext_tiny",                "family": "CNN",         "lr": 1e-4, "enabled": True},
        {"name": "swin_tiny_patch4_window7_224", "family": "Transformer", "lr": 5e-5, "enabled": True},
        {"name": "deit_small_patch16_224",       "family": "Transformer", "lr": 5e-5, "enabled": True},
        {"name": "vit_small_patch14_dinov2.lvd142m", "family": "Foundation", "lr": 2e-5, "enabled": True,
         "kwargs": {"img_size": 224}},
    ],
    "seeds": [0, 1, 2],
    "split_seed": 2026,
    "source_split": {"train": 0.7, "val": 0.1, "cal": 0.2},   # val = early stopping; cal = calibration / risk control
    "target_split": {"pool": 0.5, "test": 0.5},               # pool = unlabelled adaptation + few-label draws; test = final
    "max_images_per_class": None,                             # subsample per dataset & class (None = all)

    # ---------- deduplication ----------
    # candidate pairs from pHash/dHash are confirmed only if their 32x32 brain-cropped thumbnails correlate >= verify_corr
    "dedup": {"enabled": True, "hash_size": 8, "phash_max": 8, "dhash_max": 10, "brain_crop_threshold": 15,
              "verify_corr": 0.97},
    "min_per_class": 20,

    # ---------- training ----------
    "img_size": 224, "batch_size": 32, "num_workers": max(2, min(4, os.cpu_count() or 2)),
    "epochs": 25, "patience": 6, "warmup_epochs": 2, "min_delta": 1e-4,
    "weight_decay": 1e-4, "head_lr_mult": 10.0, "label_smoothing": 0.0, "grad_clip": 1.0, "amp": True,
    "head_hidden": 512, "head_dropout": 0.3,
    "use_dann": True, "lambda_domain": 0.2, "domain_lr_mult": 1.0,
    "use_mixstyle": True, "mixstyle_p": 0.5, "mixstyle_alpha": 0.1,
    "augment": {"hflip": 0.5, "rotation": 15, "translate": 0.1, "brightness": 0.1, "contrast": 0.1},

    # ---------- failure heads (trained post-hoc on frozen features; one run trains all of them) ----------
    "failure_heads": [
        {"name": "fh_paper", "target": "paper", "margin": 0.15, "margin_weight": 5.0,
         "diversity_weight": 0.5, "label_smoothing": 0.1},
        {"name": "fh_paper_nomargin", "target": "paper", "margin": 0.15, "margin_weight": 0.0,
         "diversity_weight": 0.5, "label_smoothing": 0.1},
        {"name": "fh_paper_nodiv", "target": "paper", "margin": 0.15, "margin_weight": 5.0,
         "diversity_weight": 0.0, "label_smoothing": 0.1},
        {"name": "fh_tcp", "target": "tcp"},
    ],
    "ablation_only_scorers": ["fh_paper_nomargin", "fh_paper_nodiv"],   # reported in the ablation table, not in main rankings
    "fh_epochs": 30, "fh_lr": 1e-3, "fh_batch": 256, "fh_aug_views": 4, "fh_error_fraction": 0.25,
    "fh_dropout": 0.3, "fh_patience": 8,

    # ---------- scoring / evaluation ----------
    "mc_dropout_passes": 30, "knn_k": 10,
    "budgets": [0.05, 0.10, 0.20, 0.30],         # review budgets for FSR@budget
    "risk_alphas": [0.01, 0.02, 0.05],           # silent-failure-rate targets
    "fsr_betas": [0.05, 0.10, 0.20],             # false-safe-rate targets
    "few_label_sizes": [25, 50, 100, 200], "few_label_draws": 50,
    "weight_clip": [0.05, 20.0], "weight_pca_dims": 64,
    "bootstrap": 1000, "bootstrap_scorers": ["msp", "energy", "mcd_mi", "knn", "fh_paper", "fh_tcp", "ens_msp", "ens_mi"],
    "binary_sensitivity_target": 0.95,           # operating point chosen on source-cal (binary tasks)
    "ece_bins": 15,

    # ---------- ablation (stage-1 components; failure-head ablations are the extra heads above) ----------
    "ablation": {"enabled": True, "shift": None, "backbone": None, "seeds": [0, 1, 2],
                 "variants": [{"name": "no_dann", "use_dann": False},
                              {"name": "no_mixstyle", "use_mixstyle": False},
                              {"name": "erm", "use_dann": False, "use_mixstyle": False}]},
    "keep_checkpoints": False,     # True keeps stage-1 weights (~100 MB per run)
}

SMOKE_OVERRIDES = {
    "max_images_per_class": 60, "seeds": [0, 1], "epochs": 1, "patience": 1, "warmup_epochs": 0,
    "fh_epochs": 2, "fh_aug_views": 2, "mc_dropout_passes": 3, "bootstrap": 20,
    "few_label_sizes": [10, 20], "few_label_draws": 3, "max_hours": 1.5,
    "ablation": {"enabled": True, "shift": None, "backbone": None, "seeds": [0],
                 "variants": [{"name": "erm", "use_dann": False, "use_mixstyle": False}]},
}
SMOKE_BACKBONES = ["efficientnet_b0", "deit_small_patch16_224"]

if RUN_MODE == "smoke":
    CFG.update(copy.deepcopy(SMOKE_OVERRIDES))
    for b in CFG["backbones"]:
        b["enabled"] = b["name"] in SMOKE_BACKBONES
    CFG["work_dir"] = CFG["work_dir"] + "_smoke"

# Optional JSON override (used for automated testing; leave unset on Kaggle)
if os.environ.get("FDSHIFT_CFG_JSON"):
    CFG.update(json.loads(Path(os.environ["FDSHIFT_CFG_JSON"]).read_text()))

WORK = Path(CFG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)
RESULTS = WORK / "results"; RUNS = WORK / "runs"; CACHE = WORK / "cache"
for _d in [RESULTS, RUNS, CACHE, RESULTS / "tables", RESULTS / "figures"]:
    _d.mkdir(parents=True, exist_ok=True)
(WORK / "config_used.json").write_text(json.dumps({"RUN_MODE": RUN_MODE, **CFG}, indent=2, default=str))
print(f"RUN_MODE={RUN_MODE} | work dir: {WORK}")
print("Backbones:", [b["name"] for b in CFG["backbones"] if b.get("enabled", True)], "| seeds:", CFG["seeds"])

RUN_MODE=paper | work dir: /kaggle/working/fdshift
Backbones: ['resnet50', 'densenet121', 'efficientnet_b0', 'convnext_tiny', 'swin_tiny_patch4_window7_224', 'deit_small_patch16_224', 'vit_small_patch14_dinov2.lvd142m'] | seeds: [0, 1, 2]


## 3 · Utilities and resume

In [3]:
# ===== 3. Utilities: logging, seeding, timing, resume =====
T_START = time.time()
LOG_FILE = WORK / "log.txt"

def log(*args):
    msg = " ".join(str(a) for a in args)
    stamp = time.strftime("%H:%M:%S")
    print(f"[{stamp}] {msg}", flush=True)
    with open(LOG_FILE, "a") as f:
        f.write(f"[{stamp}] {msg}\n")

def hours_elapsed():
    return (time.time() - T_START) / 3600.0

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    except ImportError:
        pass

def norm_name(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

def to_jsonable(o):
    if isinstance(o, dict):
        return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_jsonable(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, float):
        return None if not math.isfinite(o) else o
    if isinstance(o, np.ndarray):
        return to_jsonable(o.tolist())
    return o

def save_json(obj, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(to_jsonable(obj), indent=2))

def load_json(path):
    return json.loads(Path(path).read_text())

def safe_name(s):
    return re.sub(r"[^A-Za-z0-9_.-]", "_", str(s))

def run_dir_for(shift, bb_name, variant, seed):
    return RUNS / shift / safe_name(bb_name) / variant / f"seed{seed}"

def resume_from_previous(cfg):
    """Copy finished runs (and caches) from earlier notebook outputs into the work dir."""
    n = 0
    for rd in cfg.get("resume_dirs", []):
        rd = Path(rd)
        if not rd.exists():
            log(f"resume dir not found: {rd}"); continue
        for done in rd.glob("runs/**/done.json"):
            rel = done.parent.relative_to(rd)
            dst = WORK / rel
            if not (dst / "done.json").exists():
                shutil.copytree(done.parent, dst, dirs_exist_ok=True); n += 1
        for f in rd.glob("cache/*.csv"):
            if not (CACHE / f.name).exists():
                shutil.copy2(f, CACHE / f.name)
    if n:
        log(f"resumed {n} finished runs from previous outputs")

resume_from_previous(CFG)

## 4 · Dataset discovery and indexing
Datasets are located by **folder structure**, not by path. Br35H's unlabelled `pred/` folder is ignored automatically. The Figshare `.mat` files are converted once, and their patient IDs and tumour masks are kept.

In [4]:
# ===== 4. Dataset discovery & indexing (no hard-coded paths) =====
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def _walk(root, max_depth=6, skip=()):
    root = Path(root)
    skip = [str(Path(s)) for s in skip]
    for dirpath, dirnames, filenames in os.walk(root):
        p = Path(dirpath)
        if any(str(p).startswith(s) for s in skip):
            dirnames[:] = []; continue
        if len(p.relative_to(root).parts) >= max_depth:
            dirnames[:] = []
        dirnames.sort()
        yield p, dirnames, filenames

def _has_images(d, limit=2000):
    for i, f in enumerate(Path(d).rglob("*")):
        if f.suffix.lower() in IMG_EXT:
            return True
        if i > limit:
            return False
    return False

def find_dataset(spec, cfg):
    """Return (root, mat_files or None). Explicit root wins; otherwise search input_root by structure.
    spec["find"] may be one rule or a list of alternative rules (tried in order)."""
    rules = spec.get("find", {})
    if isinstance(rules, list):
        for r in rules:
            root, mats = find_dataset({**spec, "find": r}, cfg)
            if root is not None:
                return root, mats
        return None, None
    find = rules
    skip = [cfg["work_dir"]] + list(cfg.get("resume_dirs", []))
    if spec.get("root"):
        root = Path(spec["root"])
        if not root.exists():
            log(f"[{spec['name']}] explicit root does not exist: {root}"); return None, None
        mats = sorted(root.rglob("*.mat")) if find.get("mat") else None
        return root, mats
    base = Path(cfg["input_root"])
    if not base.exists():
        return None, None
    if find.get("mat"):
        groups = defaultdict(list)
        for p, _, files in _walk(base, skip=skip):
            n = sum(f.lower().endswith(".mat") for f in files)
            if n >= 50:
                rel = p.relative_to(base).parts
                # Kaggle mounts inputs as <slug>/... or datasets/<owner>/<slug>/... -> group by the dataset itself
                k = 3 if rel and rel[0] in ("datasets", "competitions") and len(rel) >= 3 else 1
                top = os.path.join(*rel[:k]) if rel else ""
                groups[top].append(p)
        if not groups:
            return None, None
        top = sorted(groups)[0]
        if len(groups) > 1:
            log(f"[{spec['name']}] several .mat datasets found {sorted(groups)}; using '{top}' (set 'root' to choose)")
        root = base / top
        return root, sorted(root.rglob("*.mat"))
    want = {norm_name(c) for c in find.get("child_dirs", [])}
    pc = norm_name(find["path_contains"]) if find.get("path_contains") else None
    cands = []
    for p, dirs, _ in _walk(base, skip=skip):
        names = {norm_name(d) for d in dirs}
        if want and want <= names and (pc is None or pc in norm_name(str(p.relative_to(base)))):
            first = next(d for d in dirs if norm_name(d) in want)
            if _has_images(p / first):
                cands.append(p)
    if not cands:
        return None, None
    cands.sort(key=lambda q: (len(q.parts), str(q)))
    if len(cands) > 1:
        log(f"[{spec['name']}] {len(cands)} candidate folders; using {cands[0]} (others: {[str(c) for c in cands[1:4]]})")
    return cands[0], None

def map_class(raw, spec, cfg):
    cmap = {norm_name(k): v for k, v in (spec.get("class_map") or {}).items()}
    key = norm_name(raw)
    if key in cmap:
        return cmap[key]
    return cfg["canonical_classes"].get(key)

def index_image_folder(spec, root, cfg):
    rows, skipped = [], {}
    for cdir in sorted(d for d in Path(root).iterdir() if d.is_dir()):
        files = sorted(f for f in cdir.rglob("*") if f.suffix.lower() in IMG_EXT)
        canon = map_class(cdir.name, spec, cfg)
        if canon is None:
            skipped[cdir.name] = len(files); continue
        rows += [{"path": str(f), "dataset": spec["name"], "orig_class": cdir.name, "canonical": canon,
                  "patient_id": "", "mask_path": ""} for f in files]
    if skipped:
        log(f"[{spec['name']}] ignored folders without a class mapping: {skipped}")
    return rows

def read_figshare_mat(path):
    try:
        import h5py
        with h5py.File(path, "r") as h:
            cj = h["cjdata"]
            img = np.array(cj["image"]).T
            label = int(np.array(cj["label"]).squeeze())
            pid = "".join(chr(int(c)) for c in np.array(cj["PID"]).flatten() if int(c) > 0)
            mask = np.array(cj["tumorMask"]).T if "tumorMask" in cj else None
    except (OSError, KeyError, ImportError):
        from scipy.io import loadmat
        m = loadmat(str(path), simplify_cells=True)["cjdata"]
        img = np.asarray(m["image"]); label = int(np.asarray(m["label"]).squeeze())
        pid = str(m.get("PID", "")); mask = np.asarray(m["tumorMask"]) if "tumorMask" in m else None
    return img, label, pid.strip(), mask

def _to_uint8(a):
    a = np.asarray(a, dtype=np.float32)
    lo, hi = np.percentile(a, 0.5), np.percentile(a, 99.5)
    a = np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1)
    return (a * 255).astype(np.uint8)

def index_mat_dataset(spec, mat_files, cfg):
    out = CACHE / f"converted_{spec['name']}"; out.mkdir(parents=True, exist_ok=True)
    idx_csv = CACHE / f"{spec['name']}_mat_index.csv"
    if idx_csv.exists():
        df = pd.read_csv(idx_csv, keep_default_na=False)
        if len(df) and all(Path(p).exists() for p in df["path"]):
            return df.to_dict("records")
    lim = cfg.get("max_images_per_class")
    files = list(mat_files)
    if lim:
        rng = np.random.RandomState(cfg["split_seed"]); rng.shuffle(files)
        files = files[: lim * max(1, len(spec.get("class_map") or {})) * 2]
    def convert(f):
        try:
            img, label, pid, mask = read_figshare_mat(f)
        except Exception:
            return None
        canon = map_class(str(label), spec, cfg)
        if canon is None:
            return None
        stem = Path(f).stem
        ip = out / f"{stem}.png"; Image.fromarray(_to_uint8(img)).save(ip)
        mp = ""
        if mask is not None and np.asarray(mask).size:
            mp = str(out / f"{stem}_mask.png"); Image.fromarray((np.asarray(mask) > 0).astype(np.uint8) * 255).save(mp)
        return {"path": str(ip), "dataset": spec["name"], "orig_class": canon, "canonical": canon,
                "patient_id": pid, "mask_path": mp}
    log(f"[{spec['name']}] converting {len(files)} .mat files (one-time) ...")
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=max(2, os.cpu_count() or 2)) as ex:
        res = list(ex.map(convert, files))
    rows = [r for r in res if r is not None]; bad = len(res) - len(rows)
    if bad:
        log(f"[{spec['name']}] {bad} .mat files could not be read or mapped")
    pd.DataFrame(rows).to_csv(idx_csv, index=False)
    return rows

def build_index(cfg):
    rows, roots = [], {}
    for spec in cfg["datasets"]:
        root, mats = find_dataset(spec, cfg)
        roots[spec["name"]] = str(root) if root else None
        if root is None:
            log(f"[{spec['name']}] NOT FOUND — shifts using it will be skipped"); continue
        r = index_mat_dataset(spec, mats, cfg) if mats is not None else index_image_folder(spec, root, cfg)
        log(f"[{spec['name']}] {len(r)} images from {root}")
        rows += r
    df = pd.DataFrame(rows, columns=["path", "dataset", "orig_class", "canonical", "patient_id", "mask_path"])
    lim = cfg.get("max_images_per_class")
    if lim and len(df):
        parts = [g.sample(min(len(g), lim), random_state=cfg["split_seed"])
                 for _, g in df.groupby(["dataset", "canonical"])]
        df = pd.concat(parts) if parts else df
    df = df.sort_values(["dataset", "canonical", "path"]).reset_index(drop=True)
    df["idx"] = np.arange(len(df))
    return df, roots

INDEX, DATA_ROOTS = build_index(CFG)
INDEX.to_csv(RESULTS / "dataset_index.csv", index=False)
if len(INDEX):
    print(INDEX.groupby(["dataset", "canonical"]).size().unstack(fill_value=0))
else:
    print("No datasets found under", CFG["input_root"], "- attach them via 'Add Input'.")

[06:43:53] [br35h] ignored folders without a class mapping: {'Br35H-Mask-RCNN': 801, 'pred': 60}
[06:43:53] [br35h] 3000 images from /kaggle/input/datasets/ahmedhamada0/brain-tumor-detection
[06:43:54] [btd_train] 5600 images from /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training
[06:43:54] [btd_test] 1600 images from /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing
[06:43:54] [sartaj_train] 2870 images from /kaggle/input/datasets/sartajbhuvaji/brain-tumor-classification-mri/Training
[06:43:54] [sartaj_test] 394 images from /kaggle/input/datasets/sartajbhuvaji/brain-tumor-classification-mri/Testing
[06:43:54] [figshare] converting 3065 .mat files (one-time) ...
[06:48:31] [figshare] 1 .mat files could not be read or mapped
[06:48:31] [figshare] 3064 images from /kaggle/input/datasets/ashkhagan/figshare-brain-tumor-dataset
canonical     glioma  meningioma  notumor  pituitary  tumor
dataset                                                    


## 5 · Near-duplicate detection
pHash and dHash are computed on a robust brain bounding box. Pairs within the configured Hamming distances are clustered with union-find, and patient IDs are merged into the same groups. Check `results/figures/duplicate_examples.png` visually.

In [5]:
# ===== 5. Perceptual hashing, near-duplicate clustering, patient grouping =====
def _hash_one(args):
    """pHash + dHash (64-bit each) on the brain-cropped grey image. Returns hex strings."""
    path, hs, crop_thr = args
    try:
        im = Image.open(path).convert("L")
        a = np.asarray(im)
        m = a > crop_thr                                   # robust brain bounding box (ignores stray noise pixels)
        rows, cols = np.where(m.mean(1) > 0.02)[0], np.where(m.mean(0) > 0.02)[0]
        if len(rows) > 8 and len(cols) > 8:
            a = a[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1]
        im = Image.fromarray(a)
        x = np.asarray(im.resize((hs * 4, hs * 4), Image.LANCZOS), dtype=np.float32)
        d = scipy.fft.dct(scipy.fft.dct(x, axis=0, norm="ortho"), axis=1, norm="ortho")[:hs, :hs]
        ph = (d > np.median(d)).flatten()
        y = np.asarray(im.resize((hs + 1, hs), Image.LANCZOS), dtype=np.float32)
        dh = (y[:, 1:] > y[:, :-1]).flatten()
        return np.packbits(ph).tobytes().hex(), np.packbits(dh).tobytes().hex()
    except Exception:
        return None, None

def compute_hashes(df, cfg):
    hs, thr = cfg["dedup"]["hash_size"], cfg["dedup"]["brain_crop_threshold"]
    cache_f = CACHE / f"hash_cache_h{hs}_t{thr}.csv"
    cache = pd.read_csv(cache_f, keep_default_na=False) if cache_f.exists() else pd.DataFrame(columns=["key", "ph", "dh"])
    cmap = dict(zip(cache["key"], zip(cache["ph"], cache["dh"])))
    keys = []
    for p in df["path"]:
        try:
            s = os.stat(p); keys.append(f"{p}|{s.st_size}|{int(s.st_mtime)}")
        except OSError:
            keys.append(f"{p}|missing")
    todo = [i for i, k in enumerate(keys) if k not in cmap]
    if todo:
        log(f"hashing {len(todo)} images ...")
        args = [(df["path"].iat[i], hs, thr) for i in todo]
        from concurrent.futures import ThreadPoolExecutor      # threads: robust inside notebooks
        with ThreadPoolExecutor(max_workers=max(2, (os.cpu_count() or 2))) as ex:
            res = list(ex.map(_hash_one, args, chunksize=64))
        for i, (ph, dh) in zip(todo, res):
            cmap[keys[i]] = (ph or "", dh or "")
        pd.DataFrame([{"key": k, "ph": v[0], "dh": v[1]} for k, v in cmap.items()]).to_csv(cache_f, index=False)
    ph = [cmap[k][0] for k in keys]; dh = [cmap[k][1] for k in keys]
    return ph, dh

def _thumb(args):
    """32x32 z-scored thumbnail of the brain-cropped grey image (for pixel-level verification)."""
    path, crop_thr = args
    try:
        a = np.asarray(Image.open(path).convert("L"))
        m = a > crop_thr
        rows, cols = np.where(m.mean(1) > 0.02)[0], np.where(m.mean(0) > 0.02)[0]
        if len(rows) > 8 and len(cols) > 8:
            a = a[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1]
        x = np.asarray(Image.fromarray(a).resize((32, 32), Image.BILINEAR), dtype=np.float32).ravel()
        return (x - x.mean()) / (x.std() + 1e-6)
    except Exception:
        return None

def verify_pairs(df, pairs, cfg):
    """Keep only hash-candidate pairs whose thumbnails are near-identical (removes look-alike different slices)."""
    if not len(pairs):
        return pairs, np.zeros(0)
    ids = np.unique(pairs[:, :2])
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=max(2, os.cpu_count() or 2)) as ex:
        th = list(ex.map(_thumb, [(df["path"].iat[i], cfg["dedup"]["brain_crop_threshold"]) for i in ids]))
    T = {i: t for i, t in zip(ids, th)}
    corr = np.array([float(T[i] @ T[j] / T[i].size) if T[i] is not None and T[j] is not None else 0.0
                     for i, j in pairs[:, :2]])
    keep = corr >= cfg["dedup"]["verify_corr"]
    log(f"near-duplicate verification: {keep.sum()} of {len(pairs)} hash candidates confirmed (corr >= {cfg['dedup']['verify_corr']})")
    return pairs[keep], corr[keep]

def _bits(hexes):
    return np.stack([np.unpackbits(np.frombuffer(bytes.fromhex(h), dtype=np.uint8)) for h in hexes]).astype(np.float32)

def near_duplicate_pairs(P, D, thr_p, thr_d, block=1024):
    out = []
    for i in range(0, len(P), block):
        a, b = P[i:i + block], D[i:i + block]
        dp = a @ (1 - P).T + (1 - a) @ P.T
        dd = b @ (1 - D).T + (1 - b) @ D.T
        ii, jj = np.where((dp <= thr_p) & (dd <= thr_d))
        keep = jj > ii + i
        ii, jj = ii[keep], jj[keep]
        if len(ii):
            out.append(np.stack([ii + i, jj, dp[ii, jj], dd[ii, jj]], 1))
    return np.concatenate(out).astype(int) if out else np.zeros((0, 4), int)

class DSU:
    def __init__(self, n): self.p = np.arange(n)
    def find(self, x):
        r = x
        while self.p[r] != r: r = self.p[r]
        while self.p[x] != r: self.p[x], x = r, self.p[x]
        return r
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[max(ra, rb)] = min(ra, rb)

def build_groups(df, cfg):
    df = df.copy()
    ph, dh = compute_hashes(df, cfg)
    ok = np.array([bool(a) and bool(b) for a, b in zip(ph, dh)])
    if (~ok).any():
        log(f"{(~ok).sum()} unreadable images removed")
        pd.DataFrame({"path": df.loc[~ok, "path"]}).to_csv(RESULTS / "unreadable_images.csv", index=False)
    df = df[ok].reset_index(drop=True); df["idx"] = np.arange(len(df))
    ph = [h for h, k in zip(ph, ok) if k]; dh = [h for h, k in zip(dh, ok) if k]
    n = len(df); dsu_dup = DSU(n)
    pairs = np.zeros((0, 4), int); corr = np.zeros(0)
    if cfg["dedup"]["enabled"] and n:
        pairs = near_duplicate_pairs(_bits(ph), _bits(dh), cfg["dedup"]["phash_max"], cfg["dedup"]["dhash_max"])
        pairs, corr = verify_pairs(df, pairs, cfg)
        for i, j, _, _ in pairs:
            dsu_dup.union(i, j)
    df["dup_cluster"] = [dsu_dup.find(i) for i in range(n)]
    dsu = DSU(n)                                  # groups = duplicates + same patient
    for i in range(n):
        dsu.union(i, df["dup_cluster"].iat[i])
    for _, g in df[df["patient_id"].astype(str) != ""].groupby(["dataset", "patient_id"]):
        ids = g["idx"].to_numpy()
        for k in ids[1:]:
            dsu.union(ids[0], k)
    df["group"] = [dsu.find(i) for i in range(n)]
    pr = pd.DataFrame(pairs, columns=["i", "j", "phash_dist", "dhash_dist"])
    pr["pixel_corr"] = corr
    if len(pr):
        pr["dataset_i"] = df["dataset"].to_numpy()[pr["i"]]; pr["dataset_j"] = df["dataset"].to_numpy()[pr["j"]]
        pr["class_i"] = df["canonical"].to_numpy()[pr["i"]]; pr["class_j"] = df["canonical"].to_numpy()[pr["j"]]
        pr["path_i"] = df["path"].to_numpy()[pr["i"]]; pr["path_j"] = df["path"].to_numpy()[pr["j"]]
    pr.to_csv(RESULTS / "near_duplicate_pairs.csv", index=False)
    # dataset x dataset table: number of images in A that have a near-duplicate in B
    names = sorted(df["dataset"].unique())
    mat = pd.DataFrame(0, index=names, columns=names)
    for a in names:
        for b in names:
            if len(pr):
                s = pr[(pr.dataset_i == a) & (pr.dataset_j == b)]["i"].tolist() + pr[(pr.dataset_j == a) & (pr.dataset_i == b)]["j"].tolist()
                mat.loc[a, b] = len(set(s))
    mat.to_csv(RESULTS / "tables" / "duplicates_between_datasets.csv")
    cross = pr[pr.dataset_i != pr.dataset_j] if len(pr) else pr
    if len(cross):                                   # visual check of flagged cross-dataset pairs
        k = min(12, len(cross)); sample = cross.sample(k, random_state=0)
        fig, axes = plt.subplots(k, 2, figsize=(4.2, 2.1 * k))
        axes = np.atleast_2d(axes)
        for r, (_, row) in enumerate(sample.iterrows()):
            for c_, side in enumerate(["i", "j"]):
                axes[r, c_].imshow(Image.open(row[f"path_{side}"]).convert("L"), cmap="gray")
                axes[r, c_].set_title(f"{row[f'dataset_{side}']} | {row[f'class_{side}']}" + (f" | r={row['pixel_corr']:.3f}" if side == "j" else ""),
                                      fontsize=7); axes[r, c_].axis("off")
        fig.suptitle("Flagged cross-dataset near-duplicates (check visually)", fontsize=8)
        fig.tight_layout(); fig.savefig(RESULTS / "figures" / "duplicate_examples.png", dpi=150); plt.close(fig)
    vc = df.dup_cluster.value_counts(); gc = df.group.value_counts()
    log(f"near-duplicate pairs: {len(pr)} | clusters with >1 image: {(vc > 1).sum()} | largest duplicate cluster: {vc.max()} "
        f"images | largest split group: {gc.max()} images ({100 * gc.max() / max(n, 1):.1f}% of all)")
    if gc.max() > 0.02 * n:
        log("WARNING: a very large group was formed — check results/figures/duplicate_examples.png; "
            "raise dedup.verify_corr (e.g. 0.98) if the flagged pairs are not true duplicates")
    print("Images in ROW dataset that near-duplicate an image in COLUMN dataset:\n", mat)
    return df

if len(INDEX):
    INDEX = build_groups(INDEX, CFG)
    INDEX.to_csv(RESULTS / "dataset_index.csv", index=False)

[06:48:34] hashing 16528 images ...
[06:51:30] near-duplicate verification: 22750 of 36675 hash candidates confirmed (corr >= 0.97)
[06:51:33] near-duplicate pairs: 22750 | clusters with >1 image: 3400 | largest duplicate cluster: 50 images | largest split group: 61 images (0.4% of all)
Images in ROW dataset that near-duplicate an image in COLUMN dataset:
               br35h  btd_test  btd_train  figshare  sartaj_test  sartaj_train
br35h          2232      1051       1458         1          150           386
btd_test        476       333        470       125          225           365
btd_train      1252       737       1274       940          278          2349
figshare          1       122        933       176           12          1323
sartaj_test     154       227        276        18          190           255
sartaj_train    248       385       2128      1329          255           300


## 6 · Shifts and leakage-free splits
Per shift, the pipeline:
1. drops near-duplicates whose labels conflict;
2. removes cross-domain duplicates from the configured side;
3. checks minimum class counts;
4. makes **stratified, group-aware** splits.

Source splits are train / val (early stopping) / cal (calibration). Target splits are pool (unlabelled adaptation + few-label draws) / test. An assertion guarantees that no group appears in two splits.

In [6]:
# ===== 6. Shift construction, cross-domain deduplication, leakage-free splits =====
def stratified_group_split(labels, groups, fractions, seed):
    """Assign each sample to a split; groups never straddle splits; classes stay balanced."""
    names = list(fractions)
    n = len(labels); assign = np.empty(n, dtype=object)
    remaining = np.arange(n); remaining_frac = 1.0
    for k, name in enumerate(names[:-1]):
        f = fractions[name] / remaining_frac
        n_splits = max(2, int(round(1.0 / f)))
        y, g = labels[remaining], groups[remaining]
        try:
            sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed + k)
            _, take = next(sgkf.split(np.zeros(len(remaining)), y, g))
        except ValueError:                       # too few groups: random group assignment
            rng = np.random.RandomState(seed + k)
            ug = rng.permutation(np.unique(g))
            chosen = set(ug[: max(1, int(round(len(ug) * f)))])
            take = np.where(np.isin(g, list(chosen)))[0]
        assign[remaining[take]] = name
        remaining = np.setdiff1d(remaining, remaining[take])
        remaining_frac -= fractions[name]
    assign[remaining] = names[-1]
    return assign

def build_shift(shift, index, cfg):
    task = cfg["tasks"][shift["task"]]; lab = task["labels"]
    need = list(shift["source"]) + list(shift["target"])
    have = set(index["dataset"]) if len(index) else set()
    missing = [d for d in need if d not in have]
    if missing:
        return None, {"skipped": f"missing datasets {missing}"}
    if set(shift["source"]) & set(shift["target"]):
        return None, {"skipped": "a dataset is on both sides"}
    df = index[index["dataset"].isin(need) & index["canonical"].isin(list(lab))].copy()
    df["label"] = df["canonical"].map(lab).astype(int)
    df["domain"] = np.where(df["dataset"].isin(shift["source"]), "source", "target")
    rep = {"shift": shift["name"], "task": shift["task"], "source": "+".join(shift["source"]),
           "target": "+".join(shift["target"]), "dedup_drop": shift.get("dedup_drop", "source")}
    for dom in ["source", "target"]:
        rep[f"{dom}_raw"] = int((df.domain == dom).sum())
    # 1) near-duplicate images that carry different labels are unreliable -> drop the whole duplicate cluster
    nun = df.groupby("dup_cluster")["label"].nunique()
    bad = set(nun[nun > 1].index)
    rep["label_conflict_images_removed"] = int(df["dup_cluster"].isin(bad).sum())
    df = df[~df["dup_cluster"].isin(bad)]
    # 2) groups present on both sides -> remove from the configured side
    shared = set(df.loc[df.domain == "source", "group"]) & set(df.loc[df.domain == "target", "group"])
    side = shift.get("dedup_drop", "source")
    rm = df[(df.domain == side) & df["group"].isin(shared)]
    rep["cross_domain_duplicates_removed"] = int(len(rm))
    for c, k in rm["canonical"].value_counts().items():
        rep[f"removed_{side}_{c}"] = int(k)
    df = df.drop(rm.index)
    # 3) enough images per class on both sides?
    C = len(task["class_names"])
    for dom in ["source", "target"]:
        cnt = df.loc[df.domain == dom, "label"].value_counts()
        short = [task["class_names"][c] for c in range(C) if cnt.get(c, 0) < cfg["min_per_class"]]
        if short:
            rep["skipped"] = f"{dom} has < {cfg['min_per_class']} images for {short} after deduplication"
            return None, rep
    # 4) group-aware stratified splits (fixed split_seed; training seeds vary separately)
    df["split"] = ""
    for dom, fr in [("source", cfg["source_split"]), ("target", cfg["target_split"])]:
        sub = df[df.domain == dom]
        a = stratified_group_split(sub["label"].to_numpy(), sub["group"].to_numpy(), fr, cfg["split_seed"])
        df.loc[sub.index, "split"] = [f"{dom}_{s}" for s in a]
    # leakage check: no group in two splits
    leak = df.groupby("group")["split"].nunique()
    assert (leak <= 1).all(), "group leakage across splits"
    for sp, g in df.groupby("split"):
        for c in range(C):
            rep[f"n_{sp}_{task['class_names'][c]}"] = int((g["label"] == c).sum())
    df = df.sort_values(["split", "idx"]).reset_index(drop=True)
    return df, rep

SPLIT_DIR = RESULTS / "splits"; SPLIT_DIR.mkdir(exist_ok=True)
SHIFTS, shift_reports = {}, []
for sh in CFG["shifts"]:
    if not sh.get("enabled", True):
        continue
    df_s, rep = build_shift(sh, INDEX, CFG)
    shift_reports.append(rep if rep else {"shift": sh["name"]})
    if df_s is None:
        log(f"[shift {sh['name']}] skipped: {rep.get('skipped')}"); continue
    df_s.to_csv(SPLIT_DIR / f"{sh['name']}.csv", index=False)
    SHIFTS[sh["name"]] = {"spec": sh, "df": df_s, "report": rep}
    log(f"[shift {sh['name']}] " + ", ".join(f"{k}={v}" for k, v in df_s["split"].value_counts().sort_index().items()))
pd.DataFrame(shift_reports).to_csv(RESULTS / "tables" / "data_and_deduplication.csv", index=False)
print(pd.DataFrame(shift_reports).T)

[06:51:33] [shift br35h_to_btd] source_cal=633, source_train=977, source_val=309, target_pool=798, target_test=802
[06:51:34] [shift br35h_to_sartaj] source_cal=845, source_train=1284, source_val=412, target_pool=1683, target_test=1581
[06:51:34] [shift figshare_to_sartaj] source_cal=1043, source_train=1518, source_val=498, target_pool=711, target_test=699
                                            0                         1                         2
shift                            br35h_to_btd           br35h_to_sartaj        figshare_to_sartaj
task                                   binary                    binary                   subtype
source                                  br35h                     br35h                  figshare
target                               btd_test  sartaj_train+sartaj_test  sartaj_train+sartaj_test
dedup_drop                             source                    source                    target
source_raw                               3000        

## 7 · Metrics and risk control
**Failure detection:** AURC, E-AURC, failure AUROC, AUPR-error, and FSR / precision at review budgets, with bootstrap CIs.

**Risk control (the novel contribution).** A case is auto-accepted when its confidence score s(x) ≤ τ, otherwise it is flagged for a radiologist.
- **Silent-failure rate** SFR(τ) = P(wrong ∧ accepted). It is controlled at level α with conformal risk control: choose the largest τ such that (Σᵢ Lᵢ(τ) + 1)/(n + 1) ≤ α.
- **Label-free shift adaptation:** the calibration points get likelihood-ratio weights w(x) = p_target(x)/p_source(x). These come from a domain classifier trained on frozen features using **only unlabelled target scans**. The test scan uses its own weight: (Σᵢ wᵢLᵢ(τ) + w_t)/(Σᵢ wᵢ + w_t) ≤ α (weighted conformal, Tibshirani et al. 2019).
- **False-safe rate** FSR = P(accepted | wrong) is controlled the same way, using a conformal quantile over calibration errors.
- **Baselines:** unweighted source calibration, few-label target calibration (k labelled target scans, repeated draws), and an in-sample oracle on the target test set.

In [7]:
# ===== 7. Metrics, failure-detection evaluation and conformal risk control (pure NumPy) =====
EPS = 1e-12

def softmax_np(z):
    z = np.asarray(z, dtype=np.float64); z = z - z.max(1, keepdims=True); e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def entropy_np(p):
    return -(p * np.log(p + EPS)).sum(-1)

def ece_score(probs, y, n_bins=15):
    conf, pred = probs.max(1), probs.argmax(1); acc = (pred == y)
    edges = np.linspace(0, 1, n_bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)

def classification_metrics(probs, y, C, n_bins=15):
    pred = probs.argmax(1); labels = list(range(C))
    out = {"n": int(len(y)), "n_errors": int((pred != y).sum()),
           "accuracy": accuracy_score(y, pred), "balanced_accuracy": balanced_accuracy_score(y, pred),
           "macro_f1": f1_score(y, pred, average="macro", labels=labels, zero_division=0)}
    try:
        out["auc"] = (roc_auc_score(y, probs[:, 1]) if C == 2
                      else roc_auc_score(y, probs, multi_class="ovr", average="macro", labels=labels))
    except ValueError:
        out["auc"] = float("nan")
    oh = np.eye(C)[y]
    out["brier"] = float(((probs - oh) ** 2).sum(1).mean())
    out["nll"] = float(-np.log(probs[np.arange(len(y)), y] + EPS).mean())
    out["ece"] = ece_score(probs, y, n_bins)
    if C == 2:
        out.update(binary_rates(pred, y))
    return out

def binary_rates(pred, y):
    tp = int(((pred == 1) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
    f = lambda a, b: a / b if b else float("nan")
    return {"sensitivity": f(tp, tp + fn), "specificity": f(tn, tn + fp), "ppv": f(tp, tp + fp),
            "npv": f(tn, tn + fn), "tp": tp, "tn": tn, "fp": fp, "fn": fn}

def threshold_for_sensitivity(p_pos, y, target):
    pos = np.sort(p_pos[y == 1])
    if len(pos) == 0:
        return 0.5
    k = int(np.floor((1 - target) * len(pos)))
    return float(pos[min(k, len(pos) - 1)])          # predict positive if p >= t  ->  sensitivity >= target

# ---------- failure detection (higher score = more likely wrong) ----------
def risk_coverage(scores, err):
    order = np.argsort(scores, kind="mergesort")
    e = err[order].astype(float); n = len(e)
    return np.arange(1, n + 1) / n, np.cumsum(e) / np.arange(1, n + 1)

def failure_metrics(scores, err, budgets):
    err = err.astype(bool); n, ne = len(err), int(err.sum())
    _, risk = risk_coverage(scores, err)
    _, risk_opt = risk_coverage(err.astype(float), err)
    out = {"n": n, "n_errors": ne, "error_rate": ne / n if n else float("nan"),
           "aurc": float(risk.mean()), "eaurc": float(risk.mean() - risk_opt.mean())}
    both = 0 < ne < n
    out["auroc"] = float(roc_auc_score(err, scores)) if both else float("nan")
    out["aupr_err"] = float(average_precision_score(err, scores)) if both else float("nan")
    desc = np.argsort(-scores, kind="mergesort")
    for b in budgets:
        k = int(math.ceil(b * n)); tp = int(err[desc[:k]].sum())
        out[f"fsr@{b:g}"] = 1 - tp / ne if ne else float("nan")       # errors NOT flagged within budget
        out[f"prec@{b:g}"] = tp / k if k else float("nan")
    return out

def fast_auroc(err, s):
    n1 = int(err.sum()); n0 = len(err) - n1
    if n1 == 0 or n0 == 0:
        return None
    r = st.rankdata(s)
    return float((r[err].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

def fsr_at(s, err, b):
    ne = err.sum()
    if ne == 0:
        return None
    k = int(math.ceil(b * len(err)))
    return 1 - err[np.argsort(-s, kind="mergesort")[:k]].sum() / ne

def bootstrap_ci(stat_fn, n, B, seed, alpha=0.05):
    rng = np.random.RandomState(seed); vals = []
    for _ in range(B):
        v = stat_fn(rng.randint(0, n, n))
        if v is not None and np.isfinite(v):
            vals.append(v)
    if len(vals) < max(10, B // 10):
        return float("nan"), float("nan")
    return float(np.percentile(vals, 100 * alpha / 2)), float(np.percentile(vals, 100 * (1 - alpha / 2)))

# ---------- conformal risk control ----------
def crc_threshold(scores, err, alpha, weights=None, test_weights=None):
    """Silent-failure-rate (SFR) control. A case is auto-accepted if score <= tau.
    Loss_i(tau) = err_i * 1[s_i <= tau] is monotone in tau.
    Unweighted: conformal risk control (Angelopoulos et al., 2022): (sum_i L_i + 1) / (n + 1) <= alpha.
    Weighted (covariate shift, Tibshirani et al., 2019): calibration points get likelihood-ratio weights w_i
    and the test point its own weight w_t:  (sum_i w_i L_i + w_t) / (sum_i w_i + w_t) <= alpha.
    Returns a scalar tau, or one tau per test point when test_weights is given."""
    s = np.asarray(scores, float); e = np.asarray(err, float)
    w = np.ones_like(s) if weights is None else np.asarray(weights, float)
    o = np.argsort(s, kind="mergesort"); s, e, w = s[o], e[o], w[o]
    last = np.r_[np.nonzero(np.diff(s))[0], len(s) - 1]
    cum = np.cumsum(w * e)[last]                      # weighted loss if tau = s[last[k]] (non-decreasing)
    W = w.sum()
    wt = np.atleast_1d(1.0 if test_weights is None else np.asarray(test_weights, float))
    cap = alpha * (W + wt) - wt                       # need cum <= cap
    k = np.searchsorted(cum, cap, side="right") - 1
    tau = np.where((k >= 0) & (cap >= 0), s[last[np.clip(k, 0, None)]], -np.inf)
    return float(tau[0]) if test_weights is None else tau

def fsr_threshold(scores, err, beta, weights=None, test_weights=None):
    """False-safe-rate (FSR) control: accept if score < tau (strict); P(accepted | wrong) <= beta.
    Conformal quantile over calibration ERRORS only (weighted version uses each test point's weight)."""
    s = np.asarray(scores, float); e = np.asarray(err, bool)
    w = np.ones_like(s) if weights is None else np.asarray(weights, float)
    se, we = s[e], w[e]
    wt = np.atleast_1d(1.0 if test_weights is None else np.asarray(test_weights, float))
    if len(se) == 0:
        tau = np.full(len(wt), -np.inf)
    else:
        o = np.argsort(se, kind="mergesort"); se, we = se[o], we[o]
        first = np.r_[0, np.nonzero(np.diff(se))[0] + 1]
        below = np.r_[0.0, np.cumsum(we)][first]      # error mass strictly below candidate tau (non-decreasing)
        cap = beta * (we.sum() + wt) - wt             # need below <= cap
        k = np.searchsorted(below, cap, side="right") - 1
        tau = np.where((k >= 0) & (cap >= 0), se[first[np.clip(k, 0, None)]], -np.inf)
    return float(tau[0]) if test_weights is None else tau

def evaluate_threshold(scores, err, tau, strict=False):
    err = err.astype(bool); acc = scores < tau if strict else scores <= tau
    ne = err.sum()
    return {"sfr": float((acc & err).mean()), "fsr": float((acc & err).sum() / ne) if ne else float("nan"),
            "flag_rate": float(1 - acc.mean()), "coverage": float(acc.mean()),
            "selective_risk": float(err[acc].mean()) if acc.any() else 0.0}

def density_ratio_weights(F_src, F_tgt, queries, cfg, seed):
    """w(x) = p_target(x) / p_source(x) from a balanced logistic domain classifier on frozen features.
    Uses only UNLABELLED target images.  Also returns the cross-validated domain AUC (shift magnitude)."""
    X = np.vstack([F_src, F_tgt]); d = np.r_[np.zeros(len(F_src)), np.ones(len(F_tgt))].astype(int)
    k = int(max(2, min(cfg["weight_pca_dims"], X.shape[1], len(X) - 2)))
    clf = make_pipeline(StandardScaler(), PCA(k, random_state=seed),
                        LogisticRegression(C=1.0, max_iter=5000, class_weight="balanced"))
    nf = int(min(5, np.bincount(d).min()))
    auc = float("nan")
    if nf >= 2:
        p_cv = cross_val_predict(clf, X, d, cv=StratifiedKFold(nf, shuffle=True, random_state=seed),
                                 method="predict_proba")[:, 1]
        auc = float(roc_auc_score(d, p_cv))
    clf.fit(X, d)
    lo, hi = cfg["weight_clip"]; raw = []
    for Q in queries:
        p = np.clip(clf.predict_proba(Q)[:, 1], 1e-6, 1 - 1e-6)
        raw.append(p / (1 - p))
    scale = 1.0 / max(raw[0].mean(), 1e-12)          # weights have mean 1 on the calibration set
    return [np.clip(r * scale, lo, hi) for r in raw], auc

def ess(w):
    w = np.asarray(w, float); return float(w.sum() ** 2 / (w ** 2).sum())

def risk_control_rows(scorer, cal, pool, test, w_cal, w_test, cfg, seed):
    """Compare calibration strategies for SFR and FSR guarantees on the target test set."""
    rows = []
    rng = np.random.RandomState(seed)
    draws = {m: [rng.choice(len(pool["err"]), m, replace=False) for _ in range(cfg["few_label_draws"])]
             for m in cfg["few_label_sizes"] if m <= len(pool["err"])}
    specs = [("SFR", a, crc_threshold, False) for a in cfg["risk_alphas"]] + \
            [("FSR", b, fsr_threshold, True) for b in cfg["fsr_betas"]]
    for control, level, fn, strict in specs:
        key = "sfr" if control == "SFR" else "fsr"
        def add(method, tau, extra=None):
            r = evaluate_threshold(test["s"], test["err"], tau, strict)
            tau = float(np.median(tau)) if np.ndim(tau) else tau
            rows.append({"scorer": scorer, "control": control, "level": level, "method": method, "tau": tau,
                         "realized": r[key], "violation": float(r[key] > level), **r, **(extra or {})})
        add("source", fn(cal["s"], cal["err"], level))
        add("source_weighted", fn(cal["s"], cal["err"], level, w_cal, w_test))
        add("oracle_target", fn(test["s"], test["err"], level))
        for m, ds in draws.items():
            rs = [evaluate_threshold(test["s"], test["err"], fn(pool["s"][i], pool["err"][i], level), strict) for i in ds]
            real = np.array([r[key] for r in rs], float)
            rows.append({"scorer": scorer, "control": control, "level": level, "method": f"target_{m}",
                         "tau": float("nan"), "realized": float(np.nanmean(real)),
                         "violation": float(np.nanmean(real > level)),
                         "flag_rate": float(np.mean([r["flag_rate"] for r in rs])),
                         "coverage": float(np.mean([r["coverage"] for r in rs])),
                         "selective_risk": float(np.mean([r["selective_risk"] for r in rs])),
                         "sfr": float(np.mean([r["sfr"] for r in rs])),
                         "fsr": float(np.nanmean([r["fsr"] for r in rs])), "n_draws": len(ds)})
    return rows

def evaluate_predictions(bundle, task, cfg, seed):
    """bundle[split] = {"y", "probs", "scores": {name: array}, "orig_class" (optional)}
    splits used: source_cal, target_pool, target_test.  bundle["w_cal"], bundle["domain_auc"] optional."""
    C = len(task["class_names"]); out = {"classification": {}, "failure": [], "failure_ci": [],
                                         "risk": [], "silent_by_class": [], "shift": {}}
    for sp in ["source_cal", "target_test"]:
        b = bundle[sp]; out["classification"][sp] = classification_metrics(b["probs"], b["y"], C, cfg["ece_bins"])
    if C == 2:
        t = threshold_for_sensitivity(bundle["source_cal"]["probs"][:, 1], bundle["source_cal"]["y"],
                                      cfg["binary_sensitivity_target"])
        tt = bundle["target_test"]; pred = (tt["probs"][:, 1] >= t).astype(int)
        out["operating_point"] = {"threshold": t, "target_sensitivity": cfg["binary_sensitivity_target"],
                                  "accuracy": accuracy_score(tt["y"], pred),
                                  "balanced_accuracy": balanced_accuracy_score(tt["y"], pred), **binary_rates(pred, tt["y"])}
    err = {sp: bundle[sp]["probs"].argmax(1) != bundle[sp]["y"] for sp in ["source_cal", "target_pool", "target_test"]}
    scorers = list(bundle["target_test"]["scores"])
    w_cal = bundle.get("w_cal"); w_test = bundle.get("w_test")
    if w_cal is None or w_test is None:
        w_cal, w_test = np.ones(len(bundle["source_cal"]["y"])), np.ones(len(bundle["target_test"]["y"]))
    tt_err = err["target_test"]; n_t = len(tt_err)
    for sc in scorers:
        for sp in ["source_cal", "target_test"]:
            out["failure"].append({"split": sp, "scorer": sc,
                                   **failure_metrics(bundle[sp]["scores"][sc], err[sp], cfg["budgets"])})
        s_t = bundle["target_test"]["scores"][sc]
        if 0 < tt_err.sum() < n_t and cfg["bootstrap"] > 0 and sc in cfg.get("bootstrap_scorers", scorers):
            for metric, fn in [("aurc", lambda i: risk_coverage(s_t[i], tt_err[i])[1].mean()),
                               ("fsr@0.1", lambda i: fsr_at(s_t[i], tt_err[i], 0.1)),
                               ("auroc", lambda i: fast_auroc(tt_err[i], s_t[i]))]:
                lo, hi = bootstrap_ci(fn, n_t, cfg["bootstrap"], seed)
                out["failure_ci"].append({"scorer": sc, "metric": metric, "lo": lo, "hi": hi})
        cal = {"s": bundle["source_cal"]["scores"][sc], "err": err["source_cal"]}
        pool = {"s": bundle["target_pool"]["scores"][sc], "err": err["target_pool"]}
        test = {"s": s_t, "err": tt_err}
        out["risk"] += risk_control_rows(sc, cal, pool, test, w_cal, w_test, cfg, seed)
        oc = bundle["target_test"].get("orig_class")
        if oc is not None:
            k = int(math.ceil(0.1 * n_t)); flagged = np.zeros(n_t, bool); flagged[np.argsort(-s_t, kind="mergesort")[:k]] = True
            for c in np.unique(oc):
                m = (oc == c)
                out["silent_by_class"].append({"scorer": sc, "orig_class": str(c), "n": int(m.sum()),
                                               "n_errors": int((tt_err & m).sum()),
                                               "silent_errors@0.1": int((tt_err & m & ~flagged).sum())})
    out["shift"] = {"domain_auc": bundle.get("domain_auc", float("nan")),
                    "ess_cal": ess(w_cal), "w_cal_max": float(np.max(w_cal)), "w_test_median": float(np.median(w_test)),
                    "n_cal": int(len(w_cal)),
                    "acc_drop": out["classification"]["source_cal"]["accuracy"] - out["classification"]["target_test"]["accuracy"]}
    return out

## 8 · Model
The model is a timm backbone with a dropout classification head (also used for MC-Dropout) and an optional DANN domain head behind a gradient-reversal layer. MixStyle is inserted automatically after the first stride-4 stage of any backbone that outputs NCHW feature maps; token-based transformers are skipped.

In [8]:
# ===== 8. PyTorch data pipeline and model =====
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
from timm.data import resolve_data_config

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = bool(CFG["amp"] and DEVICE.type == "cuda")
log(f"torch {torch.__version__} | timm {timm.__version__} | device {DEVICE}"
    + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else " — WARNING: no GPU, training will be very slow"))

class ImageDS(Dataset):
    """All images are loaded as single-channel and replicated to 3 channels, so colour encodings
    that differ between datasets cannot act as a shortcut."""
    def __init__(self, paths, labels, tfm):
        self.paths, self.labels, self.tfm = list(paths), np.asarray(labels, dtype=np.int64), tfm
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("L").convert("RGB")
        return self.tfm(img), int(self.labels[i]), i

def make_transforms(mean, std, cfg, train):
    s, a = cfg["img_size"], cfg["augment"]
    ops = [T.Resize((s, s))]
    if train:
        ops += [T.RandomHorizontalFlip(a["hflip"]), T.RandomRotation(a["rotation"]),
                T.RandomAffine(0, translate=(a["translate"], a["translate"])),
                T.ColorJitter(brightness=a["brightness"], contrast=a["contrast"])]
    ops += [T.ToTensor(), T.Normalize(mean, std)]
    return T.Compose(ops)

def _seed_worker(worker_id):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s); random.seed(s)

def make_loader(df, tfm, cfg, shuffle, seed, drop_last=False):
    g = torch.Generator(); g.manual_seed(int(seed))
    drop_last = drop_last and len(df) > cfg["batch_size"]
    return DataLoader(ImageDS(df["path"].values, df["label"].values, tfm), batch_size=cfg["batch_size"],
                      shuffle=shuffle, num_workers=cfg["num_workers"], pin_memory=DEVICE.type == "cuda",
                      drop_last=drop_last, generator=g, worker_init_fn=_seed_worker)

class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, g):
        return -ctx.alpha * g, None

class MixStyle(nn.Module):
    """Zhou et al., ICLR 2021: mixes channel-wise feature statistics between samples of a batch."""
    def __init__(self, p=0.5, alpha=0.1, eps=1e-6):
        super().__init__(); self.p, self.alpha, self.eps = p, alpha, eps
    def forward(self, x):
        if not self.training or x.size(0) < 2 or torch.rand(1).item() > self.p:
            return x
        B = x.size(0)
        mu = x.mean(dim=[2, 3], keepdim=True)
        sig = (x.var(dim=[2, 3], keepdim=True) + self.eps).sqrt()
        mu, sig = mu.detach(), sig.detach()
        lam = torch.distributions.Beta(self.alpha, self.alpha).sample((B, 1, 1, 1)).to(x.device, x.dtype)
        perm = torch.randperm(B, device=x.device)
        return (x - mu) / sig * (sig * lam + sig[perm] * (1 - lam)) + (mu * lam + mu[perm] * (1 - lam))

class FDNet(nn.Module):
    """Backbone -> features z; classification head (with dropout, used for MC-Dropout);
    optional domain head behind a gradient-reversal layer (DANN)."""
    def __init__(self, bb, num_classes, cfg, use_dann, use_mixstyle):
        super().__init__()
        self.backbone = timm.create_model(bb["name"], pretrained=True, num_classes=0, **bb.get("kwargs", {}))
        d, h, p = self.backbone.num_features, cfg["head_hidden"], cfg["head_dropout"]
        self.num_features = d
        self.cls_head = nn.Sequential(nn.Dropout(p), nn.Linear(d, h), nn.LayerNorm(h), nn.GELU(),
                                      nn.Dropout(p), nn.Linear(h, num_classes))
        self.dom_head = (nn.Sequential(nn.Linear(d, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 2))
                         if use_dann else None)
        self.mixstyle_at = None
        if use_mixstyle:
            self._attach_mixstyle(cfg)

    def _attach_mixstyle(self, cfg):
        """Insert MixStyle after the first stride>=4 stage, only where it outputs NCHW feature maps
        (all CNNs incl. ConvNeXt); token-based transformers are skipped automatically."""
        infos = getattr(self.backbone, "feature_info", None)
        if not infos:
            return
        infos = infos.info if hasattr(infos, "info") else list(infos)
        cands = [f for f in infos if f.get("reduction", 0) >= 4 and f.get("module")]
        mods = dict(self.backbone.named_modules())
        if not cands or cands[0]["module"] not in mods:
            return
        name, chans = cands[0]["module"], cands[0]["num_chs"]
        seen = {}
        hdl = mods[name].register_forward_hook(lambda m, i, o: seen.__setitem__("shape", tuple(o.shape) if torch.is_tensor(o) else None))
        was = self.backbone.training; self.backbone.eval()
        with torch.no_grad():
            self.backbone(torch.zeros(1, 3, cfg["img_size"], cfg["img_size"]))
        hdl.remove(); self.backbone.train(was)
        shp = seen.get("shape")
        if shp and len(shp) == 4 and shp[1] == chans:
            self.mixstyle = MixStyle(cfg["mixstyle_p"], cfg["mixstyle_alpha"])
            mods[name].register_forward_hook(lambda m, i, o: self.mixstyle(o))
            self.mixstyle_at = name

    def forward(self, x, grl_alpha=None):
        f = self.backbone(x)
        logits = self.cls_head(f)
        dom = None
        if self.dom_head is not None and grl_alpha is not None:
            dom = self.dom_head(GradReverse.apply(f, grl_alpha))
        return f, logits, dom

class FailureHead(nn.Module):
    """Learned failure predictor on frozen features (LayerNorm MLP + scaled linear residual)."""
    def __init__(self, d, dropout=0.3):
        super().__init__()
        h = max(32, d // 4)
        self.mlp = nn.Sequential(nn.Linear(d, h), nn.LayerNorm(h), nn.LeakyReLU(0.2), nn.Dropout(dropout),
                                 nn.Linear(h, h // 2), nn.LayerNorm(h // 2), nn.LeakyReLU(0.2), nn.Dropout(dropout),
                                 nn.Linear(h // 2, 1))
        self.res = nn.Linear(d, 1)
        nn.init.xavier_uniform_(self.res.weight, gain=0.1); nn.init.zeros_(self.res.bias)
    def forward(self, z):
        return (self.mlp(z) + 0.1 * self.res(z)).squeeze(1)

[06:51:35] torch 2.11.0+cu128 | timm 1.0.29 | device cuda (Tesla T4)


## 9 · Training and confidence scoring functions
Stage 1 trains on source data, plus unlabelled target data when DANN is on. Mixed precision, warm-up and cosine schedule, early stopping on **source validation only**.

The model is then frozen in eval mode, so BatchNorm statistics are never updated with target images. Failure heads are trained post-hoc on frozen features, using clean plus augmented views of the source training set to obtain more error examples. Every scoring function is evaluated on the **same classifier**.

In [9]:
# ===== 9. Stage-1 training, feature extraction, failure heads, confidence scoring functions =====
def _cycle(loader):
    while True:
        for b in loader:
            yield b

@torch.no_grad()
def eval_loss_acc(model, loader):
    model.eval(); tot, n, corr = 0.0, 0, 0
    for xb, yb, _ in loader:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            _, logits, _ = model(xb)
        tot += F.cross_entropy(logits.float(), yb, reduction="sum").item()
        corr += (logits.argmax(1) == yb).sum().item(); n += len(yb)
    return tot / max(n, 1), corr / max(n, 1)

def train_stage1(model, dl_src, dl_val, dl_tgt, bb, cfg, use_dann):
    model.to(DEVICE)
    cls_params = [p for n, p in model.named_parameters() if n.startswith("cls_head.")]
    dom_params = [p for n, p in model.named_parameters() if n.startswith("dom_head.")]
    groups = [{"params": model.backbone.parameters(), "lr": bb["lr"]},
              {"params": cls_params, "lr": bb["lr"] * cfg["head_lr_mult"]}]
    if dom_params:                                  # domain head at base LR: keeps the adversarial game stable
        groups.append({"params": dom_params, "lr": bb["lr"] * cfg.get("domain_lr_mult", 1.0)})
    opt = torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])
    spe = len(dl_src); total = max(1, cfg["epochs"] * spe); warm = cfg["warmup_epochs"] * spe
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except (AttributeError, TypeError):                     # older PyTorch
        scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    tgt_it = _cycle(dl_tgt) if use_dann else None
    best_loss, best_state, best_ep, bad, step, hist = float("inf"), None, -1, 0, 0, []
    for ep in range(cfg["epochs"]):
        model.train(); t0 = time.time(); run_loss, nb = 0.0, 0
        for xb, yb, _ in dl_src:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            alpha = 2.0 / (1.0 + math.exp(-10.0 * step / total)) - 1.0           # Ganin et al. schedule
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                if use_dann:
                    xt = next(tgt_it)[0].to(DEVICE, non_blocking=True)
                    _, logits, dom = model(torch.cat([xb, xt]), grl_alpha=alpha)
                    ns = xb.size(0)
                    dlab = torch.cat([torch.zeros(ns), torch.ones(xt.size(0))]).long().to(DEVICE)
                    loss = (F.cross_entropy(logits[:ns].float(), yb, label_smoothing=cfg["label_smoothing"])
                            + cfg["lambda_domain"] * F.cross_entropy(dom.float(), dlab))
                else:
                    _, logits, _ = model(xb)
                    loss = F.cross_entropy(logits.float(), yb, label_smoothing=cfg["label_smoothing"])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            scaler.step(opt); scaler.update(); sched.step(); step += 1
            run_loss += loss.item(); nb += 1
        vl, va = eval_loss_acc(model, dl_val)
        hist.append({"epoch": ep + 1, "train_loss": run_loss / max(nb, 1), "val_loss": vl, "val_acc": va,
                     "seconds": time.time() - t0})
        if vl < best_loss - cfg["min_delta"]:
            best_loss, best_ep, bad = vl, ep + 1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        log(f"    epoch {ep + 1:02d} train {run_loss / max(nb, 1):.4f} | val loss {vl:.4f} acc {va:.4f} | {time.time() - t0:.0f}s")
        if bad >= cfg["patience"]:
            break
    model.load_state_dict(best_state)
    model.eval()
    return hist, best_ep

@torch.no_grad()
def extract(model, df, tfm, cfg, seed=0):
    model.eval(); torch.manual_seed(seed)
    feats, logits = [], []
    for xb, _, _ in make_loader(df, tfm, cfg, shuffle=False, seed=seed):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            f, lg, _ = model(xb.to(DEVICE, non_blocking=True))
        feats.append(f.float().cpu()); logits.append(lg.float().cpu())
    return torch.cat(feats).numpy(), torch.cat(logits).numpy()

@torch.no_grad()
def mc_dropout(model, feats, passes, seed, chunk=4096):
    """MC-Dropout on the classification head (features are deterministic)."""
    head = model.cls_head; head.train(); torch.manual_seed(seed)
    X = torch.from_numpy(feats).float()
    P = np.zeros((passes, len(feats), head[-1].out_features), dtype=np.float32)
    for t in range(passes):
        for i in range(0, len(X), chunk):
            P[t, i:i + chunk] = torch.softmax(head(X[i:i + chunk].to(DEVICE)).float(), 1).cpu().numpy()
    head.eval()
    mean = P.mean(0)
    return mean, entropy_np(mean) - entropy_np(P).mean(0)

@torch.no_grad()
def knn_score(bank, query, k, chunk=2048):
    """Deep k-NN distance (Sun et al., ICML 2022): minus cosine similarity to the k-th nearest source-train feature."""
    B = F.normalize(torch.from_numpy(bank).float().to(DEVICE), dim=1); out = []
    k = int(min(k, len(bank)))
    for i in range(0, len(query), chunk):
        Q = F.normalize(torch.from_numpy(query[i:i + chunk]).float().to(DEVICE), dim=1)
        out.append(-(Q @ B.T).topk(k, dim=1).values[:, -1].cpu())
    return torch.cat(out).numpy()

def paper_failure_targets(P, y, ls):
    conf, err = P.max(1), (P.argmax(1) != y)
    t = (1 - conf) + 0.5 * err
    t = np.where(err, np.maximum(t, 0.65), t)
    t = np.where(~err & (conf > 0.8), np.minimum(t, 0.35), t)
    return np.clip(t, 0, 1) * (1 - ls) + 0.5 * ls

def train_failure_head(spec, Ftr, Ptr, ytr, Fva, Pva, yva, cfg, seed):
    torch.manual_seed(seed); rng = np.random.RandomState(seed)
    err_tr = Ptr.argmax(1) != ytr; err_va = Pva.argmax(1) != yva
    t = Ptr[np.arange(len(ytr)), ytr] if spec["target"] == "tcp" else paper_failure_targets(Ptr, ytr, spec.get("label_smoothing", 0.0))
    head = FailureHead(Ftr.shape[1], cfg["fh_dropout"]).to(DEVICE)
    opt = torch.optim.AdamW(head.parameters(), lr=cfg["fh_lr"], weight_decay=1e-4)
    n, ne = len(ytr), int(err_tr.sum()); frac = cfg["fh_error_fraction"]
    p = np.where(err_tr, frac / max(ne, 1), (1 - frac) / max(n - ne, 1)) if 0 < ne < n * frac else np.ones(n) / n
    p = p / p.sum()
    Xtr, Ttr, Etr = (torch.from_numpy(Ftr).float().to(DEVICE), torch.from_numpy(t).float().to(DEVICE),
                     torch.from_numpy(err_tr).to(DEVICE))
    Xva = torch.from_numpy(Fva).float().to(DEVICE)
    def score(X):
        head.eval()
        with torch.no_grad():
            s = torch.sigmoid(head(X)).cpu().numpy()
        return 1 - s if spec["target"] == "tcp" else s
    best, best_state, bad = -float("inf"), None, 0
    use_auc = err_va.sum() >= 3 and (~err_va).sum() >= 3
    for ep in range(cfg["fh_epochs"]):
        head.train(); idx = rng.choice(n, n, replace=True, p=p); tl = 0.0
        for i in range(0, n, cfg["fh_batch"]):
            b = torch.from_numpy(idx[i:i + cfg["fh_batch"]]).to(DEVICE)
            out = head(Xtr[b]); sig = torch.sigmoid(out)
            if spec["target"] == "tcp":
                loss = F.mse_loss(sig, Ttr[b])
            else:
                loss = F.binary_cross_entropy_with_logits(out, Ttr[b])
                if spec.get("diversity_weight", 0) > 0 and len(b) > 1:
                    loss = loss + spec["diversity_weight"] * (-torch.log(sig.std() + 1e-6))
                e = Etr[b]
                if spec.get("margin_weight", 0) > 0 and e.any() and (~e).any():
                    loss = loss + spec["margin_weight"] * F.relu(spec["margin"] - (sig[e].mean() - sig[~e].mean()))
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); tl += loss.item()
        s_va = score(Xva)
        crit = roc_auc_score(err_va, s_va) if use_auc else -tl
        if crit > best + 1e-6:
            best, bad = crit, 0
            best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg["fh_patience"]:
                break
    head.load_state_dict(best_state)
    info = {"name": spec["name"], "n_train": n, "n_train_errors": ne, "n_val_errors": int(err_va.sum()),
            "selection": "val_auroc" if use_auc else "train_loss", "best": float(best)}
    return (lambda Fx: score(torch.from_numpy(Fx).float().to(DEVICE))), info

def logsumexp_np(z):
    m = z.max(1, keepdims=True); return (m + np.log(np.exp(z - m).sum(1, keepdims=True))).squeeze(1)

def base_scores(logits):
    P = softmax_np(logits)
    return {"msp": 1 - P.max(1), "entropy": entropy_np(P), "maxlogit": -logits.max(1), "energy": -logsumexp_np(logits)}

## 10 · Run all experiments
Runs go shift by shift, then seed, then backbone, so complete comparisons accumulate early. Finished runs are skipped, and the loop stops before `max_hours`. Per-sample predictions are saved, so ensembles and figures need no retraining.

In [10]:
# ===== 10. One experiment run (shift x backbone x variant x seed) and the main loop =====
EVAL_SPLITS = ["source_val", "source_cal", "target_pool", "target_test"]

def run_one(shift_name, bb, seed, variant):
    vname = variant.get("name", "full")
    rd = run_dir_for(shift_name, bb["name"], vname, seed)
    if (rd / "done.json").exists():
        return "cached"
    rd.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    S = SHIFTS[shift_name]; task = CFG["tasks"][S["spec"]["task"]]; C = len(task["class_names"])
    use_dann = variant.get("use_dann", CFG["use_dann"]); use_ms = variant.get("use_mixstyle", CFG["use_mixstyle"])
    log(f"=== RUN {shift_name} | {bb['name']} | {vname} | seed {seed} | DANN={use_dann} MixStyle={use_ms}")
    set_seed(seed)
    model = FDNet(bb, C, CFG, use_dann, use_ms)
    dc = resolve_data_config({}, model=model.backbone)
    tf_train = make_transforms(dc["mean"], dc["std"], CFG, True)
    tf_test = make_transforms(dc["mean"], dc["std"], CFG, False)
    parts = {k: g.reset_index(drop=True) for k, g in S["df"].groupby("split")}
    dl_src = make_loader(parts["source_train"], tf_train, CFG, shuffle=True, seed=seed, drop_last=True)
    dl_val = make_loader(parts["source_val"], tf_test, CFG, shuffle=False, seed=seed)
    dl_tgt = make_loader(parts["target_pool"], tf_train, CFG, shuffle=True, seed=seed + 7, drop_last=True) if use_dann else None
    hist, best_ep = train_stage1(model, dl_src, dl_val, dl_tgt, bb, CFG, use_dann)
    t_train = time.time() - t0
    # --- frozen-model inference (eval mode everywhere: BN statistics, MixStyle and dropout are fixed)
    feats, logits = {}, {}
    for k in ["source_train"] + EVAL_SPLITS:
        feats[k], logits[k] = extract(model, parts[k], tf_test, CFG, seed)
    y = {k: parts[k]["label"].to_numpy() for k in parts}
    Fv, Lv, Yv = [feats["source_train"]], [logits["source_train"]], [y["source_train"]]
    for v in range(max(0, CFG["fh_aug_views"] - 1)):                       # augmented views => more training errors
        f, l = extract(model, parts["source_train"], tf_train, CFG, seed=seed * 1000 + v + 1)
        Fv.append(f); Lv.append(l); Yv.append(y["source_train"])
    Ftr, Ptr, Ytr = np.concatenate(Fv), softmax_np(np.concatenate(Lv)), np.concatenate(Yv)
    scores = {k: base_scores(logits[k]) for k in EVAL_SPLITS}
    fh_info = []
    for spec in CFG["failure_heads"]:
        fn, info = train_failure_head(spec, Ftr, Ptr, Ytr, feats["source_val"], softmax_np(logits["source_val"]),
                                      y["source_val"], CFG, seed)
        fh_info.append(info)
        for k in EVAL_SPLITS:
            scores[k][spec["name"]] = fn(feats[k])
    for k in EVAL_SPLITS:
        mean, mi = mc_dropout(model, feats[k], CFG["mc_dropout_passes"], seed)
        scores[k]["mcd_msp"] = 1 - mean.max(1); scores[k]["mcd_mi"] = mi
        scores[k]["knn"] = knn_score(feats["source_train"], feats[k], CFG["knn_k"])
    (w_cal, w_test), dom_auc = density_ratio_weights(feats["source_cal"], feats["target_pool"],
                                                     [feats["source_cal"], feats["target_test"]], CFG, seed)
    bundle = {k: {"y": y[k], "probs": softmax_np(logits[k]), "scores": scores[k]} for k in EVAL_SPLITS}
    bundle["target_test"]["orig_class"] = np.array([str(v) for v in parts["target_test"]["orig_class"]], dtype=str)
    bundle["w_cal"], bundle["w_test"], bundle["domain_auc"] = w_cal, w_test, dom_auc
    metrics = evaluate_predictions(bundle, task, CFG, seed)
    # --- save per-sample predictions (used for ensembles and figures) and metrics
    arrays = {"w_cal": w_cal, "w_test": w_test}
    for k in EVAL_SPLITS:
        arrays[f"{k}__y"] = y[k]; arrays[f"{k}__probs"] = bundle[k]["probs"].astype(np.float32)
        arrays[f"{k}__idx"] = parts[k]["idx"].to_numpy()
        for sc, v in scores[k].items():
            arrays[f"{k}__score__{sc}"] = np.asarray(v, np.float32)
    arrays["target_test__orig_class"] = bundle["target_test"]["orig_class"]
    np.savez_compressed(rd / "predictions.npz", **arrays)
    save_json(metrics, rd / "metrics.json"); save_json(hist, rd / "history.json")
    if CFG["keep_checkpoints"]:
        torch.save(model.state_dict(), rd / "model.pt")
    meta = {"shift": shift_name, "task": S["spec"]["task"], "backbone": bb["name"], "family": bb["family"],
            "variant": vname, "seed": seed, "use_dann": use_dann, "use_mixstyle": use_ms,
            "mixstyle_at": model.mixstyle_at, "params_m": sum(p.numel() for p in model.parameters()) / 1e6,
            "feature_dim": model.num_features, "best_epoch": best_ep, "epochs_run": len(hist),
            "train_minutes": t_train / 60, "total_minutes": (time.time() - t0) / 60, "failure_heads": fh_info,
            "data_mean": list(dc["mean"]), "data_std": list(dc["std"]),
            "torch": torch.__version__, "timm": timm.__version__,
            "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"}
    save_json(meta, rd / "done.json")
    tt = metrics["classification"]["target_test"]
    msp = next(r for r in metrics["failure"] if r["split"] == "target_test" and r["scorer"] == "msp")
    log(f"    done in {meta['total_minutes']:.1f} min | target acc {tt['accuracy']:.3f} bacc {tt['balanced_accuracy']:.3f}"
        f" | MSP AURC {msp['aurc']:.4f} FSR@10% {msp['fsr@0.1']:.3f} | domain AUC {dom_auc:.3f}")
    del model; torch.cuda.empty_cache() if DEVICE.type == "cuda" else None
    return "done"

def planned_runs():
    bbs = [b for b in CFG["backbones"] if b.get("enabled", True)]
    plan = [(s, b, seed, {"name": "full"}) for s in SHIFTS for seed in CFG["seeds"] for b in bbs]
    ab = CFG.get("ablation", {})
    if ab.get("enabled") and SHIFTS:
        s = ab.get("shift") or next(iter(SHIFTS))
        b = next((x for x in bbs if x["name"] == ab.get("backbone")), bbs[0] if bbs else None)
        if s in SHIFTS and b is not None:
            plan += [(s, b, seed, v) for v in ab["variants"] for seed in ab["seeds"]]
    return plan

PLAN = planned_runs()
log(f"{len(PLAN)} planned runs; {sum((run_dir_for(s, b['name'], v['name'], sd) / 'done.json').exists() for s, b, sd, v in PLAN)} already finished")
durations = []
for s, b, sd, v in PLAN:
    if (run_dir_for(s, b["name"], v["name"], sd) / "done.json").exists():
        continue
    est = np.median(durations) if durations else 0.3
    if hours_elapsed() + est > CFG["max_hours"]:
        log(f"time budget reached ({hours_elapsed():.2f} h) — remaining runs will continue in the next session"); break
    t1 = time.time()
    try:
        run_one(s, b, sd, v)
    except Exception as e:
        import traceback
        log(f"!!! run failed: {s} {b['name']} {v['name']} seed{sd}: {e}")
        if any(t in str(e).lower() for t in ["connect", "resolve", "huggingface", "urlopen", "offline"]):
            log("    hint: pretrained weights could not be downloaded — enable Internet in the notebook settings")
        (run_dir_for(s, b["name"], v["name"], sd)).mkdir(parents=True, exist_ok=True)
        (run_dir_for(s, b["name"], v["name"], sd) / "error.txt").write_text(traceback.format_exc())
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    durations.append((time.time() - t1) / 3600)

[06:51:35] 72 planned runs; 0 already finished
[06:51:35] === RUN br35h_to_btd | resnet50 | full | seed 0 | DANN=True MixStyle=True


model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

[06:51:51]     epoch 01 train 0.7591 | val loss 0.4773 acc 0.8091 | 13s
[06:52:00]     epoch 02 train 0.5013 | val loss 0.3560 acc 0.8479 | 10s
[06:52:10]     epoch 03 train 0.4127 | val loss 0.4225 acc 0.8350 | 10s
[06:52:20]     epoch 04 train 0.3128 | val loss 0.3114 acc 0.8673 | 10s
[06:52:30]     epoch 05 train 0.2648 | val loss 0.2820 acc 0.9061 | 11s
[06:52:40]     epoch 06 train 0.2706 | val loss 0.2999 acc 0.8997 | 10s
[06:52:50]     epoch 07 train 0.2186 | val loss 0.2880 acc 0.9223 | 10s
[06:53:00]     epoch 08 train 0.2276 | val loss 0.2143 acc 0.9353 | 10s
[06:53:11]     epoch 09 train 0.1866 | val loss 0.3072 acc 0.9126 | 11s
[06:53:21]     epoch 10 train 0.1653 | val loss 0.2496 acc 0.9450 | 10s
[06:53:32]     epoch 11 train 0.1832 | val loss 0.2153 acc 0.9515 | 10s
[06:53:42]     epoch 12 train 0.1742 | val loss 0.2483 acc 0.9385 | 11s
[06:53:54]     epoch 13 train 0.1820 | val loss 0.2341 acc 0.9515 | 12s
[06:54:05]     epoch 14 train 0.1594 | val loss 0.2768 acc 0.941

model.safetensors: reconstructing file:   0%|          |  0.00B / 32.3MB            

...

model.safetensors: downloading bytes:           |  0.00B            

[07:08:12]     epoch 01 train 0.6729 | val loss 0.3694 acc 0.8479 | 10s
[07:08:21]     epoch 02 train 0.2571 | val loss 0.1574 acc 0.9644 | 9s
[07:08:30]     epoch 03 train 0.2502 | val loss 0.2454 acc 0.9320 | 9s
[07:08:40]     epoch 04 train 0.4416 | val loss 0.1015 acc 0.9773 | 10s
[07:08:50]     epoch 05 train 0.5045 | val loss 0.1328 acc 0.9612 | 10s
[07:08:59]     epoch 06 train 0.4797 | val loss 0.1931 acc 0.9644 | 9s
[07:09:08]     epoch 07 train 0.4808 | val loss 0.1351 acc 0.9741 | 9s
[07:09:17]     epoch 08 train 0.4290 | val loss 0.0866 acc 0.9838 | 9s
[07:09:27]     epoch 09 train 0.4912 | val loss 0.1870 acc 0.9644 | 10s
[07:09:36]     epoch 10 train 0.5514 | val loss 0.2121 acc 0.9676 | 9s
[07:09:45]     epoch 11 train 0.5178 | val loss 0.1236 acc 0.9806 | 9s
[07:09:55]     epoch 12 train 0.4578 | val loss 0.1005 acc 0.9773 | 9s
[07:10:05]     epoch 13 train 0.5051 | val loss 0.1148 acc 0.9676 | 10s
[07:10:14]     epoch 14 train 0.4894 | val loss 0.3916 acc 0.9417 | 9s
[

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

[07:10:57]     epoch 01 train 0.7439 | val loss 0.2324 acc 0.9256 | 13s
[07:11:09]     epoch 02 train 0.9574 | val loss 0.1120 acc 0.9612 | 12s
[07:11:20]     epoch 03 train 1.4033 | val loss 0.1934 acc 0.9482 | 12s
[07:11:32]     epoch 04 train 1.2489 | val loss 0.3258 acc 0.9288 | 11s
[07:11:44]     epoch 05 train 1.1186 | val loss 0.2306 acc 0.9644 | 12s
[07:11:55]     epoch 06 train 0.9850 | val loss 0.2250 acc 0.9579 | 11s
[07:12:06]     epoch 07 train 0.8367 | val loss 0.2851 acc 0.9417 | 11s
[07:12:18]     epoch 08 train 0.6858 | val loss 0.2740 acc 0.9612 | 12s
[07:12:49]     done in 2.1 min | target acc 0.771 bacc 0.831 | MSP AURC 0.0821 FSR@10% 0.783 | domain AUC 0.943
[07:12:49] === RUN br35h_to_btd | resnet50 | full | seed 1 | DANN=True MixStyle=True
[07:13:01]     epoch 01 train 0.7146 | val loss 0.4585 acc 0.7994 | 12s
[07:13:13]     epoch 02 train 0.5048 | val loss 0.3507 acc 0.8673 | 11s
[07:13:24]     epoch 03 train 0.3999 | val loss 0.6784 acc 0.7508 | 12s
[07:13:35] 

## 11 · Aggregate: ensembles, tables, statistics, figures
All paper numbers come from here. Tables go to `results/tables/*.csv|.tex`, figures to `results/figures/`, and the auto-generated methods text to `results/methods_summary.md`.

In [11]:
# ===== 11. Aggregation: ensembles, tables (CSV + LaTeX), statistics, figures, methods summary =====
TAB, FIG = RESULTS / "tables", RESULTS / "figures"
KEYS = ["shift", "task", "backbone", "family", "variant", "seed"]
EVAL_SPLITS = ["source_val", "source_cal", "target_pool", "target_test"]

def collect_runs():
    recs = []
    for done in sorted(RUNS.glob("*/*/*/*/done.json")):
        mp = done.parent / "metrics.json"
        if mp.exists():
            recs.append((load_json(done), load_json(mp), done.parent))
    return recs

def build_ensembles(recs):
    groups = defaultdict(list)
    for meta, _, d in recs:
        if meta["variant"] == "full" and meta["seed"] != "ens":
            groups[(meta["shift"], meta["backbone"])].append((meta, d))
    out = []
    for (s, b), lst in groups.items():
        if len(lst) < 2:
            continue
        lst = sorted(lst, key=lambda x: x[0]["seed"]); seeds = [m["seed"] for m, _ in lst]
        od = lst[0][1].parent / "ensemble"
        if (od / "done.json").exists() and load_json(od / "done.json").get("members") == seeds:
            out.append((load_json(od / "done.json"), load_json(od / "metrics.json"), od)); continue
        zs = [np.load(d / "predictions.npz") for _, d in lst]
        bundle = {}
        for k in EVAL_SPLITS:
            y0 = zs[0][f"{k}__y"]
            if not all(np.array_equal(z[f"{k}__y"], y0) for z in zs):
                log(f"ensemble {s}/{b}: split '{k}' differs between seeds — skipped"); bundle = None; break
            P = np.stack([z[f"{k}__probs"].astype(np.float64) for z in zs]); m = P.mean(0)
            bundle[k] = {"y": y0, "probs": m, "scores": {"ens_msp": 1 - m.max(1), "ens_entropy": entropy_np(m),
                                                          "ens_mi": entropy_np(m) - entropy_np(P).mean(0)}}
        if bundle is None:
            continue
        bundle["target_test"]["orig_class"] = zs[0]["target_test__orig_class"]
        bundle["w_cal"] = np.mean([z["w_cal"] for z in zs], 0); bundle["w_test"] = np.mean([z["w_test"] for z in zs], 0)
        mets = [mm for (mt, mm, dd) in recs if dd in [d for _, d in lst]]
        bundle["domain_auc"] = float(np.nanmean([mm["shift"]["domain_auc"] for mm in mets])) if mets else float("nan")
        task = CFG["tasks"][lst[0][0]["task"]]
        metrics = evaluate_predictions(bundle, task, CFG, seed=0)
        od.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(od / "predictions.npz", **{f"{k}__score__{sc}": v for k in EVAL_SPLITS
                                                         for sc, v in bundle[k]["scores"].items()},
                            **{f"{k}__y": bundle[k]["y"] for k in EVAL_SPLITS},
                            **{f"{k}__probs": bundle[k]["probs"].astype(np.float32) for k in EVAL_SPLITS})
        meta = {**{k: lst[0][0][k] for k in ["shift", "task", "backbone", "family"]},
                "variant": "full", "seed": "ens", "members": seeds}
        save_json(metrics, od / "metrics.json"); save_json(meta, od / "done.json")
        out.append((meta, metrics, od))
    return out

def long_tables(recs):
    T = defaultdict(list)
    for meta, m, _ in recs:
        key = {k: meta.get(k) for k in KEYS}
        for sp, c in m["classification"].items():
            T["cls"].append({**key, "split": sp, **c})
        if "operating_point" in m:
            T["op"].append({**key, **m["operating_point"]})
        T["fd"] += [{**key, **r} for r in m["failure"]]
        T["ci"] += [{**key, **r} for r in m["failure_ci"]]
        T["risk"] += [{**key, **r} for r in m["risk"]]
        T["sil"] += [{**key, **r} for r in m["silent_by_class"]]
        T["shift"].append({**key, **m["shift"]})
    return {k: pd.DataFrame(v) for k, v in T.items()}

def mean_sd(df, keys, cols, digits=3):
    g = df.groupby(keys, sort=False)
    m, s, n = g[cols].mean(), g[cols].std(ddof=1), g.size()
    fmt = m.copy().astype(object)
    for c in cols:
        fmt[c] = [(f"{a:.{digits}f} ± {b:.{digits}f}" if np.isfinite(b) else f"{a:.{digits}f}") if np.isfinite(a) else "–"
                  for a, b in zip(m[c], s[c])]
    fmt["n_seeds"] = n; m["n_seeds"] = n
    return m.reset_index(), fmt.reset_index()

def save_table(df, name, caption=None):
    if df is None or len(df) == 0:
        return
    df.to_csv(TAB / f"{name}.csv", index=False)
    try:
        (TAB / f"{name}.tex").write_text(df.to_latex(index=False, escape=True, caption=caption, label=f"tab:{name}"))
    except Exception:
        pass

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for r, i in enumerate(o):
        run = max(run, min(1.0, (len(p) - r) * p[i])); adj[i] = run
    return adj

def numeric_seed(df):
    return df[df["seed"].apply(lambda v: isinstance(v, (int, np.integer)))]

BASE_SCORERS_ORDER = ["msp", "entropy", "maxlogit", "energy", "mcd_msp", "mcd_mi", "knn"]

def aggregate():
    recs = collect_runs()
    recs = [r for r in recs if r[0]["seed"] != "ens"]
    if not recs:
        log("no finished runs to aggregate yet"); return None
    ens = build_ensembles(recs)
    L = long_tables(recs + ens)
    cls, fd, risk = L["cls"], L["fd"], L["risk"]
    full = lambda d: d[d["variant"] == "full"]
    # ---- classification
    c = full(cls[cls.split == "target_test"])
    cols = [k for k in ["accuracy", "balanced_accuracy", "macro_f1", "auc", "ece", "brier", "nll",
                        "sensitivity", "specificity", "n_errors"] if k in c]
    _, f1 = mean_sd(numeric_seed(c), ["shift", "backbone", "family"], cols)
    e = c[c.seed == "ens"]
    if len(e):
        e2 = e[["shift", "backbone", "family"] + cols].copy(); e2["backbone"] = e2["backbone"] + " (ensemble)"
        e2[cols] = e2[cols].round(3).astype(str); e2["n_seeds"] = "ens"
        f1 = pd.concat([f1, e2], ignore_index=True)
    save_table(f1, "classification_target_test", "Target-domain classification (mean ± SD over seeds).")
    cs = full(cls[cls.split == "source_cal"])
    if len(cs):
        _, fsrc = mean_sd(numeric_seed(cs), ["shift", "backbone"], [k for k in ["accuracy", "balanced_accuracy", "auc", "ece"] if k in cs])
        save_table(fsrc, "classification_source_heldout", "Source held-out classification (in-domain reference).")
    if len(L["op"]):
        op = full(numeric_seed(L["op"]))
        _, fop = mean_sd(op, ["shift", "backbone"], ["threshold", "accuracy", "balanced_accuracy", "sensitivity", "specificity"])
        save_table(fop, "binary_operating_point", "Binary operating point chosen on source calibration data, applied on target.")
    # ---- failure detection
    ft = full(fd[fd.split == "target_test"])
    mcols = [k for k in ["aurc", "eaurc", "auroc", "aupr_err", "fsr@0.1", "fsr@0.2", "prec@0.1"] if k in ft]
    num, fmt = mean_sd(numeric_seed(ft), ["shift", "backbone", "scorer"], mcols, 4)
    save_table(fmt, "failure_detection_target", "Failure detection on the target test set (mean ± SD over seeds).")
    fe = ft[ft.seed == "ens"]
    if len(fe):
        save_table(fe[["shift", "backbone", "scorer"] + mcols].round(4), "failure_detection_ensembles")
    pooled = numeric_seed(ft).groupby(["shift", "scorer"])[mcols].agg(["mean", "std"]).round(4)
    pooled.columns = [f"{a}_{b}" for a, b in pooled.columns]; save_table(pooled.reset_index(), "failure_detection_pooled_over_backbones")
    fs = full(fd[fd.split == "source_cal"])
    if len(fs):
        deg = numeric_seed(fs).groupby(["shift", "scorer"])[["aurc", "auroc", "fsr@0.1"]].mean().add_suffix("_source").join(
              numeric_seed(ft).groupby(["shift", "scorer"])[["aurc", "auroc", "fsr@0.1"]].mean().add_suffix("_target"))
        save_table(deg.round(4).reset_index(), "failure_detection_source_vs_target")
    # ---- statistics: Friedman ranks + Wilcoxon vs MSP (blocks = backbone x seed)
    stat_rows, rank_rows = [], []
    main = lambda d: d[~d["scorer"].isin(CFG.get("ablation_only_scorers", []))]
    for s, g in numeric_seed(main(ft)).groupby("shift"):
        piv = g.pivot_table(index=["backbone", "seed"], columns="scorer", values="aurc").dropna(axis=1, how="any")
        if piv.shape[0] >= 2 and piv.shape[1] >= 3:
            ranks = piv.rank(axis=1).mean().sort_values()
            try:
                fp = st.friedmanchisquare(*[piv[c].values for c in piv.columns]).pvalue
            except Exception:
                fp = float("nan")
            rank_rows += [{"shift": s, "scorer": k, "mean_rank_aurc": v, "friedman_p": fp, "n_blocks": piv.shape[0]} for k, v in ranks.items()]
            if "msp" in piv:
                tmp = []
                for sc in piv.columns:
                    if sc == "msp":
                        continue
                    d = piv[sc] - piv["msp"]
                    try:
                        p = st.wilcoxon(d.values).pvalue if (d != 0).sum() >= 5 else float("nan")
                    except Exception:
                        p = float("nan")
                    tmp.append({"shift": s, "scorer": sc, "median_aurc_minus_msp": float(d.median()),
                                "n_better_than_msp": int((d < 0).sum()), "n_blocks": int(len(d)), "p_wilcoxon": p})
                ps = np.array([t["p_wilcoxon"] for t in tmp], float); ok = np.isfinite(ps)
                adj = np.full_like(ps, np.nan); adj[ok] = holm(ps[ok]) if ok.any() else adj[ok]
                for t, a in zip(tmp, adj):
                    t["p_holm"] = a
                stat_rows += tmp
    save_table(pd.DataFrame(rank_rows), "scorer_ranks_friedman", "Mean rank of each scoring function by AURC (lower is better).")
    save_table(pd.DataFrame(stat_rows), "scorer_vs_msp_wilcoxon", "Paired Wilcoxon tests against MSP (Holm-corrected).")
    # ---- CNN vs non-CNN (backbone-level means, honest n)
    fam_rows = []
    msp_t = numeric_seed(ft[ft.scorer == "msp"]); acc_t = numeric_seed(c)
    for s in msp_t["shift"].unique():
        a = msp_t[msp_t["shift"] == s].groupby(["backbone", "family"])[["aurc", "fsr@0.1"]].mean().reset_index()
        b = acc_t[acc_t["shift"] == s].groupby(["backbone", "family"])[["balanced_accuracy"]].mean().reset_index()
        a = a.merge(b, on=["backbone", "family"])
        cnn = a[a.family == "CNN"]; oth = a[a.family != "CNN"]
        for col in ["balanced_accuracy", "aurc", "fsr@0.1"]:
            p = (st.mannwhitneyu(cnn[col], oth[col]).pvalue if len(cnn) >= 2 and len(oth) >= 2 else float("nan"))
            fam_rows.append({"shift": s, "metric": col, "cnn_median": cnn[col].median(), "non_cnn_median": oth[col].median(),
                             "n_cnn": len(cnn), "n_non_cnn": len(oth), "p_mannwhitney_backbone_level": p})
    save_table(pd.DataFrame(fam_rows), "cnn_vs_transformer", "CNN vs transformer/foundation backbones (backbone-level).")
    # ---- risk control
    if len(risk):
        rk = numeric_seed(full(risk))
        rsum = (rk.groupby(["shift", "scorer", "control", "level", "method"])
                  .agg(realized_mean=("realized", "mean"), realized_sd=("realized", "std"),
                       violation_rate=("violation", "mean"), flag_rate=("flag_rate", "mean"),
                       selective_risk=("selective_risk", "mean"), n_runs=("realized", "size")).round(4).reset_index())
        save_table(rsum, "risk_control_summary", "Realised risk on the target test set for each calibration strategy.")
        re_ = risk[(risk.seed == "ens") & (risk.variant == "full")]
        if len(re_):
            save_table(re_.drop(columns=["task", "variant"]).round(4), "risk_control_ensembles")
    # ---- ablation
    ab = CFG.get("ablation", {})
    if ab.get("enabled"):
        vv = numeric_seed(cls[cls.split == "target_test"])
        s0 = ab.get("shift") or (vv["shift"].iloc[0] if len(vv) else None)
        bbs = [b["name"] for b in CFG["backbones"] if b.get("enabled", True)]
        b0 = ab.get("backbone") or (bbs[0] if bbs else None)
        sub_c = vv[(vv["shift"] == s0) & (vv.backbone == b0) & vv.seed.isin(ab["seeds"])]
        sub_f = numeric_seed(fd[(fd.split == "target_test") & (fd["shift"] == s0) & (fd.backbone == b0) & fd.seed.isin(ab["seeds"])])
        if len(sub_c):
            a1 = sub_c.groupby("variant")[["accuracy", "balanced_accuracy", "ece"]].agg(["mean", "std"])
            a2 = sub_f[sub_f.scorer.isin(["msp", "fh_paper", "fh_tcp", "knn"])].pivot_table(
                index="variant", columns="scorer", values=["aurc", "fsr@0.1"], aggfunc="mean")
            a1.columns = [f"{x}_{y}" for x, y in a1.columns]; a2.columns = [f"{x}_{y}" for x, y in a2.columns]
            save_table(a1.join(a2).round(4).reset_index(), "ablation_stage1", f"Ablation on {s0} / {b0}.")
        fh = numeric_seed(ft[ft.scorer.str.startswith("fh_")])
        if len(fh):
            _, fhf = mean_sd(fh, ["shift", "scorer"], ["aurc", "auroc", "fsr@0.1"], 4)
            save_table(fhf, "ablation_failure_heads", "Failure-head loss ablation (all backbones and seeds).")
    # ---- silent failures by original class
    if len(L["sil"]):
        sl = numeric_seed(full(L["sil"])).groupby(["shift", "scorer", "orig_class"])[["n", "n_errors", "silent_errors@0.1"]].sum().reset_index()
        sl["silent_share_of_errors"] = (sl["silent_errors@0.1"] / sl["n_errors"].replace(0, np.nan)).round(3)
        save_table(sl, "silent_failures_by_original_class")
    # ---- shift magnitude vs failure detection
    sh = numeric_seed(full(L["shift"]))
    if len(sh):
        mm = sh.merge(msp_t[["shift", "backbone", "seed", "aurc", "fsr@0.1"]], on=["shift", "backbone", "seed"], how="left")
        save_table(mm.drop(columns=["task", "variant"]).round(4), "shift_magnitude_per_run")
        cr = []
        for col in ["acc_drop", "aurc", "fsr@0.1"]:
            ok = mm[["domain_auc", col]].dropna()
            if len(ok) >= 5:
                rho, p = st.spearmanr(ok["domain_auc"], ok[col]); cr.append({"x": "domain_auc", "y": col, "spearman_rho": rho, "p": p, "n_runs": len(ok)})
        save_table(pd.DataFrame(cr), "shift_magnitude_correlations")
    make_figures(L, ft)
    write_methods_summary(recs, L)
    return L

def _npz_scores(d, split):
    z = np.load(d / "predictions.npz")
    y, P = z[f"{split}__y"], z[f"{split}__probs"]
    sc = {k.split("__score__")[1]: z[k] for k in z.files if k.startswith(f"{split}__score__")}
    return y, P, sc

def make_figures(L, ft):
    plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "font.size": 9})
    cls = L["cls"]
    for s in ft["shift"].unique():
        g = numeric_seed(ft[(ft["shift"] == s) & ~ft["scorer"].isin(CFG.get("ablation_only_scorers", []))])
        if not len(g):
            continue
        # AURC heatmap backbone x scorer
        hm = g.pivot_table(index="backbone", columns="scorer", values="aurc", aggfunc="mean")
        fig, ax = plt.subplots(figsize=(1.0 + 0.75 * hm.shape[1], 0.9 + 0.45 * hm.shape[0]))
        im = ax.imshow(hm.values, cmap="viridis_r", aspect="auto")
        ax.set_xticks(range(hm.shape[1])); ax.set_xticklabels(hm.columns, rotation=45, ha="right")
        ax.set_yticks(range(hm.shape[0])); ax.set_yticklabels(hm.index)
        for i in range(hm.shape[0]):
            for j in range(hm.shape[1]):
                ax.text(j, i, f"{hm.values[i, j]:.3f}", ha="center", va="center", fontsize=7, color="w")
        fig.colorbar(im, ax=ax, label="AURC (lower = better)"); ax.set_title(f"{s}: AURC on target test")
        fig.tight_layout(); fig.savefig(FIG / f"aurc_heatmap_{s}.png"); plt.close(fig)
        # risk-coverage curves for the backbone with best mean target balanced accuracy (seed with lowest id)
        cc = numeric_seed(cls[(cls["shift"] == s) & (cls.split == "target_test") & (cls.variant == "full")])
        if len(cc):
            bb = cc.groupby("backbone")["balanced_accuracy"].mean().idxmax()
            sd = sorted(cc[cc.backbone == bb]["seed"].unique())[0]
            d = run_dir_for(s, bb, "full", sd)
            if (d / "predictions.npz").exists():
                y, P, sc = _npz_scores(d, "target_test"); err = P.argmax(1) != y
                fig, ax = plt.subplots(figsize=(5.2, 3.6))
                for name in [k for k in ["msp", "energy", "mcd_mi", "knn", "fh_paper", "fh_tcp"] if k in sc]:
                    cov, rk = risk_coverage(sc[name], err); ax.plot(cov, rk, label=f"{name} (AURC {rk.mean():.3f})", lw=1.2)
                cov, rk = risk_coverage(err.astype(float), err); ax.plot(cov, rk, "k--", lw=0.8, label="oracle")
                ax.set_xlabel("Coverage (fraction auto-accepted)"); ax.set_ylabel("Risk (error rate among accepted)")
                ax.set_title(f"{s} | {bb} | seed {sd}"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
                fig.tight_layout(); fig.savefig(FIG / f"risk_coverage_{s}.png"); plt.close(fig)
        # risk control: realised SFR vs target, and the few-label curve
        rk = L["risk"]
        if len(rk):
            r = numeric_seed(rk[(rk["shift"] == s) & (rk.variant == "full") & (rk.scorer == "msp") & (rk.control == "SFR")])
            if len(r):
                meth = [m for m in ["source", "source_weighted"] + [f"target_{k}" for k in CFG["few_label_sizes"]] + ["oracle_target"] if m in set(r.method)]
                levels = sorted(r.level.unique())
                fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
                w = 0.8 / max(1, len(meth))
                for j, m in enumerate(meth):
                    q = r[r.method == m].groupby("level")
                    mu = q["realized"].mean().reindex(levels).to_numpy(); sd = q["realized"].std().reindex(levels).fillna(0).to_numpy()
                    axes[0].bar(np.arange(len(levels)) + j * w, mu, w, yerr=[np.minimum(sd, mu), sd], label=m, capsize=2)
                    axes[1].bar(np.arange(len(levels)) + j * w, q["flag_rate"].mean().reindex(levels), w, label=m)
                for k, a in enumerate(levels):
                    axes[0].hlines(a, k - w / 2, k + (len(meth) - 0.5) * w, colors="k", linestyles="--", lw=0.9)
                for ax, t in zip(axes, ["Realised silent-failure rate on target (dashed = target α)", "Flag rate (radiologist review workload)"]):
                    ax.set_xticks(np.arange(len(levels)) + (len(meth) - 1) * w / 2); ax.set_xticklabels([f"α={a:g}" for a in levels]); ax.set_title(t, fontsize=9)
                axes[0].set_ylim(bottom=0); axes[1].set_ylim(0, 1.05)
                h, l = axes[0].get_legend_handles_labels()
                fig.legend(h, l, loc="lower center", ncol=min(len(meth), 7), fontsize=7, frameon=False)
                fig.suptitle(f"{s}: silent-failure-rate control with MSP (mean ± SD over runs)", fontsize=10)
                fig.tight_layout(rect=(0, 0.08, 1, 1)); fig.savefig(FIG / f"risk_control_{s}.png"); plt.close(fig)
                lv = levels[len(levels) // 2]; few = r[(r.level == lv) & r.method.str.startswith("target_")].copy()
                if len(few):
                    few["m"] = few.method.str.replace("target_", "").astype(int)
                    q = few.groupby("m")[["violation", "flag_rate"]].mean()
                    fig, ax = plt.subplots(figsize=(4.8, 3.2))
                    ax.plot(q.index, q["violation"], "o-", label="P(realised SFR > α)")
                    ax.plot(q.index, q["flag_rate"], "s-", label="flag rate")
                    for m, ls in [("source", ":"), ("source_weighted", "-.")]:
                        z = r[(r.level == lv) & (r.method == m)]
                        if len(z):
                            ax.axhline(z["violation"].mean(), ls=ls, color="gray", label=f"{m}: violation")
                    ax.set_xlabel("Labelled target scans used for calibration"); ax.set_title(f"{s}: α = {lv:g}")
                    ax.legend(fontsize=7); ax.grid(alpha=0.3); fig.tight_layout(); fig.savefig(FIG / f"few_label_{s}.png"); plt.close(fig)

def write_methods_summary(recs, L):
    lines = ["# Auto-generated methods & results summary", "",
             f"Generated: {time.strftime('%Y-%m-%d %H:%M')} | RUN_MODE = {RUN_MODE}", "", "## Datasets found", ""]
    for k, v in DATA_ROOTS.items():
        lines.append(f"- **{k}**: {v or 'not attached'}")
    lines += ["", "## Data after deduplication and splitting", ""]
    try:
        lines.append(pd.read_csv(TAB / "data_and_deduplication.csv").T.to_markdown())
    except Exception:
        lines.append("(see tables/data_and_deduplication.csv)")
    lines += ["", "## Backbones", "", "| backbone | family | params (M) | feature dim | MixStyle inserted at | mean best epoch | mean minutes/run |",
              "|---|---|---|---|---|---|---|"]
    meta = pd.DataFrame([m for m, _, _ in recs if m.get("variant") == "full"])
    if len(meta):
        for b, g in meta.groupby("backbone", sort=False):
            lines.append(f"| {b} | {g.family.iloc[0]} | {g.params_m.iloc[0]:.1f} | {g.feature_dim.iloc[0]} | "
                         f"{g.mixstyle_at.iloc[0] or '— (not applicable)'} | {g.best_epoch.mean():.1f} | {g.total_minutes.mean():.1f} |")
        lines += ["", f"Software: torch {meta.torch.iloc[0]}, timm {meta.timm.iloc[0]}; hardware: {meta.gpu.iloc[0]}."]
    keys = ["img_size", "batch_size", "epochs", "patience", "warmup_epochs", "weight_decay", "head_lr_mult", "label_smoothing",
            "grad_clip", "head_hidden", "head_dropout", "use_dann", "lambda_domain", "use_mixstyle", "mixstyle_p", "mixstyle_alpha",
            "augment", "seeds", "split_seed", "source_split", "target_split", "dedup", "failure_heads", "fh_epochs", "fh_lr",
            "fh_aug_views", "fh_error_fraction", "mc_dropout_passes", "knn_k", "budgets", "risk_alphas", "fsr_betas",
            "few_label_sizes", "few_label_draws", "weight_clip", "weight_pca_dims", "bootstrap", "binary_sensitivity_target"]
    lines += ["", "## Hyper-parameters actually used", ""] + [f"- `{k}`: {json.dumps(CFG[k], default=str)}" for k in keys]
    lines += ["", "Backbone learning rates: " + ", ".join(f"{b['name']}={b['lr']:g}" for b in CFG["backbones"] if b.get("enabled", True)),
              "", "## Where each result lives", "",
              "- Classification: `tables/classification_target_test.csv`, operating point: `tables/binary_operating_point.csv`",
              "- Failure detection: `tables/failure_detection_target.csv`, statistics: `scorer_ranks_friedman.csv`, `scorer_vs_msp_wilcoxon.csv`",
              "- Risk control (novel contribution): `tables/risk_control_summary.csv`, figures `risk_control_*.png`, `few_label_*.png`",
              "- Ablations: `ablation_stage1.csv`, `ablation_failure_heads.csv`; shift magnitude: `shift_magnitude_correlations.csv`",
              "- Duplicates: `duplicates_between_datasets.csv`, `../near_duplicate_pairs.csv`"]
    (RESULTS / "methods_summary.md").write_text("\n".join(lines))

L_TABLES = aggregate()
for name in ["classification_target_test", "scorer_ranks_friedman", "risk_control_summary"]:
    p = TAB / f"{name}.csv"
    if p.exists():
        print(f"\n### {name}"); print(pd.read_csv(p).head(40).to_string(index=False))
zp = WORK / "fdshift_results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in RESULTS.rglob("*"):
        if f.is_file():
            z.write(f, f.relative_to(WORK))
    for f in [WORK / "config_used.json", LOG_FILE]:
        if f.exists():
            z.write(f, f.relative_to(WORK))
log(f"results zipped to {zp}")


### classification_target_test
             shift                                    backbone      family      accuracy balanced_accuracy      macro_f1           auc           ece         brier           nll   sensitivity   specificity          n_errors n_seeds
      br35h_to_btd                               convnext_tiny         CNN 0.404 ± 0.087     0.595 ± 0.059 0.393 ± 0.100 0.978 ± 0.002 0.519 ± 0.181 1.021 ± 0.357 1.985 ± 1.081 0.201 ± 0.118 0.989 ± 0.006  478.000 ± 70.000       3
      br35h_to_btd                      deit_small_patch16_224 Transformer 0.648 ± 0.284     0.755 ± 0.194 0.646 ± 0.281 0.979 ± 0.015 0.270 ± 0.219 0.524 ± 0.423 0.815 ± 0.634 0.533 ± 0.381 0.977 ± 0.010 282.333 ± 228.088       3
      br35h_to_btd                                 densenet121         CNN 0.854 ± 0.019     0.886 ± 0.012 0.832 ± 0.018 0.965 ± 0.008 0.101 ± 0.029 0.244 ± 0.040 0.537 ± 0.171 0.820 ± 0.031 0.952 ± 0.027  117.333 ± 15.044       3
      br35h_to_btd                          